# India Hydrological Outlook — End-to-End Automation Pipeline

**Input** (from `Hydrologic_Outlook/Input/`): seven monthly parameter files for one date stamp `YYYY_MM_DD`.

**Output** (written back to `Hydrologic_Outlook/Output/`):
1. `All_Maps/<Parameter>/<Column>.png` — every column rendered as its own map
2. `Dashboards/<Parameter>_dashboard.png` — 9-panel composite per parameter
3. `Hydrolook_<YYYY_MM_DD>.pdf` — the 9-page monthly outlook with LLM-written summaries

**Pipeline stages:**
1. Mount Drive, install deps, resolve paths and the forecast date
2. Load India administrative boundaries (DataMeet → GADM → user upload)
3. Render all per-column maps and the 7 composite dashboards using the **exact dashboard code from `IHO_Maps_Dashboard.ipynb`** (lifted verbatim — same layout, same legend, same extreme-panel framing)
4. Compute per-region directional histograms from the 7 raw files (the dashboards' vocabulary, not raw means)
5. Load Qwen3.5-4B (non-thinking) and have it write 12 paragraphs grounded in those statistics, each within a strict word budget so the LaTeX template doesn't overflow
6. Patch the LaTeX template with new paragraphs + dashboards + static images, run XeLaTeX twice, copy the PDF to Drive

**One thing to edit before running:** `FORECAST_DATE_STR` in Section 1, if it should be anything other than the latest snapshot in the Input folder.

**Runtime:** *Runtime → Change runtime type → T4 GPU*. End-to-end run on a fresh T4 takes ~25–35 minutes (~10 min for TeX Live, ~5 min for the Qwen3.5-4B download, ~10–15 min for the actual pipeline).

## 1 · Mount Drive, set paths, resolve the date

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, shutil, subprocess, sys, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# === Drive layout ===
DRIVE_ROOT      = Path('/content/drive/MyDrive/Hydrologic_Outlook')
INPUT_DIR       = DRIVE_ROOT / 'Input'
OUTPUT_BASE     = DRIVE_ROOT / 'Output'
STATIC_IMG_DIR  = DRIVE_ROOT / 'PDF_images'           # static logos, basin map, photos
HYDROQA_ZIP     = Path('/content/drive/MyDrive/hydroqa.zip')   # the QnA chatbot package

ALL_MAPS_DIR    = OUTPUT_BASE / 'All_Maps'
DASHBOARDS_DIR  = OUTPUT_BASE / 'Dashboards'

# === Forecast date ===
# Set to None to auto-pick the latest YYYY_MM_DD found in INPUT_DIR; otherwise hard-code e.g. '2026_02_28'.
FORECAST_DATE_STR = None

DATE_RE = re.compile(r'_(\d{4}_\d{2}_\d{2})$')

def autodetect_date(input_dir: Path) -> str:
    found = set()
    for p in input_dir.iterdir():
        m = DATE_RE.search(p.name)
        if m:
            found.add(m.group(1))
    if not found:
        raise FileNotFoundError(f'No files matching *_YYYY_MM_DD in {input_dir}')
    return sorted(found)[-1]

if FORECAST_DATE_STR is None:
    FORECAST_DATE_STR = autodetect_date(INPUT_DIR)

# Verify all 7 parameter files exist for this date
PREFIXES = ['P', 'T', 'sm', 'ro', 'ET', 'Q', 'Station_Q']
missing = [f'{p}_{FORECAST_DATE_STR}' for p in PREFIXES
           if not (INPUT_DIR / f'{p}_{FORECAST_DATE_STR}').exists()]
if missing:
    raise FileNotFoundError(f'Missing parameter files for {FORECAST_DATE_STR}: {missing}')

# Recompute the calendar labels we'll use throughout
YEAR, MONTH, DAY = map(int, FORECAST_DATE_STR.split('_'))
BASE = datetime(YEAR, MONTH, 1)

def step_month(base, n):
    mm, yy = base.month + n, base.year
    while mm <= 0:  mm += 12; yy -= 1
    while mm > 12: mm -= 12; yy += 1
    return datetime(yy, mm, 1)

LABELS = {
    'current':   BASE.strftime('%B'),
    'forecast':  step_month(BASE, +1).strftime('%B'),
    'prev1':     step_month(BASE, -1).strftime('%B'),
    'prev2':     step_month(BASE, -2).strftime('%B'),
    'prev3':     step_month(BASE, -3).strftime('%B'),
    'prev4':     step_month(BASE, -4).strftime('%B'),
    'last_year': f"{BASE.strftime('%B')} ({BASE.year-1}*)",
    'current_year':  BASE.year,
    'forecast_year': step_month(BASE, +1).year,
}

# Clean output dirs for a fresh run
for d in [ALL_MAPS_DIR, DASHBOARDS_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

print(f'Forecast date    : {FORECAST_DATE_STR}  (current = {LABELS["current"]} {YEAR}, forecast = {LABELS["forecast"]} {LABELS["forecast_year"]})')
print(f'Input dir        : {INPUT_DIR}')
print(f'All maps out     : {ALL_MAPS_DIR}')
print(f'Dashboards out   : {DASHBOARDS_DIR}')
print(f'Static images    : {STATIC_IMG_DIR}')
print(f'HydroQA package  : {HYDROQA_ZIP}  (exists={HYDROQA_ZIP.exists()})')
print(f'Static images present: {STATIC_IMG_DIR.exists()}')

In [ ]:
# Install Python deps and TeX Live (XeLaTeX + Carlito font).
# First time on a fresh Colab this takes ~5–10 min for the texlive packages;
# subsequent runs in the same session are instant.
%pip install -q geopandas shapely pyshp matplotlib numpy pandas scipy requests pyarrow accelerate
%pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git@main"

# XeLaTeX + the Carlito font + a handful of LaTeX packages the template uses
import subprocess
TEX_PACKAGES = (
    'texlive-xetex texlive-fonts-recommended texlive-fonts-extra '
    'texlive-latex-extra fonts-crosextra-carlito'
)
print('Installing TeX Live …')
subprocess.run(f'apt-get -qq update', shell=True, check=False)
subprocess.run(f'apt-get -qq install -y {TEX_PACKAGES}', shell=True, check=True)

# Sanity
out = subprocess.run(['xelatex', '--version'], capture_output=True, text=True)
print('xelatex:', out.stdout.splitlines()[0] if out.stdout else 'NOT FOUND')
out = subprocess.run(['fc-list', ':family'], capture_output=True, text=True)
print('Carlito installed:', 'Carlito' in (out.stdout or ''))

## 2 · Load India administrative boundaries

Same DataMeet → GADM cascade as the original notebook so India is drawn with PoK + Aksai Chin included when DataMeet is available. The DataMeet zip is cached to Drive after the first download so subsequent monthly runs reuse it instantly.

In [ ]:
import io, zipfile, urllib.request
import geopandas as gpd
from shapely.geometry import box

BOUNDARY_DIR = Path('/content/india_boundaries')
BOUNDARY_DIR.mkdir(exist_ok=True)
DATAMEET_ZIP_CACHE = DRIVE_ROOT / 'datameet_maps.zip'

def _download(url, dest):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=300) as r:
        with open(dest, 'wb') as f:
            f.write(r.read())

def _find_member(zip_path, suffix):
    with zipfile.ZipFile(zip_path) as z:
        matches = [m for m in z.namelist() if m.endswith(suffix)]
    if not matches:
        raise FileNotFoundError(f'No *{suffix} inside {zip_path}')
    return sorted(matches, key=len)[0]

def _load_datameet():
    if not DATAMEET_ZIP_CACHE.exists():
        print(f'  Downloading DataMeet (~50 MB, cached after first run) …')
        DATAMEET_ZIP_CACHE.parent.mkdir(parents=True, exist_ok=True)
        _download('https://github.com/datameet/maps/archive/refs/heads/master.zip',
                  DATAMEET_ZIP_CACHE)
    cm = _find_member(DATAMEET_ZIP_CACHE, 'Country/india-composite.geojson')
    sm = _find_member(DATAMEET_ZIP_CACHE, 'Survey-of-India-Index-Maps/Boundaries/India-States.shp')
    dm = _find_member(DATAMEET_ZIP_CACHE, 'Survey-of-India-Index-Maps/Boundaries/India-Districts-2011Census.shp')
    cg = gpd.read_file(f'zip://{DATAMEET_ZIP_CACHE}!{cm}').to_crs(4326)
    sg = gpd.read_file(f'zip://{DATAMEET_ZIP_CACHE}!{sm}').to_crs(4326)
    dg = gpd.read_file(f'zip://{DATAMEET_ZIP_CACHE}!{dm}').to_crs(4326)
    return cg, sg, dg

country_gdf = states_gdf = districts_gdf = None
try:
    country_gdf, states_gdf, districts_gdf = _load_datameet()
    src_label = 'DataMeet / Survey-of-India (includes PoK + Aksai Chin)'
except Exception as e:
    print(f'  ⚠️ DataMeet failed: {e}\n  Falling back to GADM 4.1 (NB: excludes PoK + Aksai Chin)')
    zp = BOUNDARY_DIR / 'gadm41_IND_shp.zip'
    if not zp.exists():
        _download('https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IND_shp.zip', zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(BOUNDARY_DIR)
    country_gdf   = gpd.read_file(BOUNDARY_DIR / 'gadm41_IND_0.shp')
    states_gdf    = gpd.read_file(BOUNDARY_DIR / 'gadm41_IND_1.shp')
    districts_gdf = gpd.read_file(BOUNDARY_DIR / 'gadm41_IND_2.shp')
    src_label = 'GADM 4.1 (excludes PoK + Aksai Chin)'

for g in (country_gdf, states_gdf, districts_gdf):
    if g is not None and g.crs is None:
        g.set_crs(epsg=4326, inplace=True)

INDIA_UNION = (country_gdf if country_gdf is not None and len(country_gdf) else states_gdf).unary_union
print(f'✓ Boundaries loaded: {src_label}')
print(f'  states={len(states_gdf)}  districts={len(districts_gdf)}')

## 3 · Per-parameter colormaps + configuration

Lifted directly from `IHO_Maps_Dashboard.ipynb`. Bin boundaries, colour ramps, and category labels match the published indiahydrolook.in dashboards.

In [ ]:
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

plt.rcParams['font.family'] = 'DejaVu Sans'

# ---------- 1 · Rainfall percentile (P) — 20 bins, 0..100 in steps of 5 ----------
P_BOUNDS = list(range(0, 101, 5))                                  # 21 edges → 20 bins
P_COLORS = [
    '#9E0142','#B01546','#CE374D','#E2524A','#F36C43','#F98F53',
    '#FDB264','#FECD7B','#FEE6A4','#FFF7B1','#F8FCB4','#EAF79F',
    '#D0EC9C','#AFDDA3','#89D1A4','#65C1A5','#48A1B3','#3881BA',
    '#505CAA','#5E4FA2',
]
P_CATEGORY_LABELS = ['Exceptional low','Extreme low','Very low','Low','Below average',
                     'Above average','High','Very high','Extreme high','Exceptional high']

# ---------- 2 · Temperature anomaly (T) — 20 bins, –4.5 .. +4.5 step 0.5 ----------
T_BOUNDS = [-1e9, -4.5, -4, -3.5, -3, -2.5, -2, -1.5, -1, -0.5,
                 0,  0.5,  1,  1.5,  2,  2.5,  3,  3.5,  4,  4.5, 1e9]
T_COLORS = [
    '#000000','#0A3278','#0F4BA5','#1E6EC8','#3CA0F0','#50B4FA',
    '#82D2FF','#A0F0FF','#C8FAFF','#E6FFFF',
    '#FFFADC','#FFE878','#FFC03C','#FFA000','#FF6000',
    '#FF3200','#E11400','#C00000','#A50000','#800026',
]
T_CATEGORY_LABELS = ['Exceptional cold','Extreme cold','Very cold','Medium cold','Less cold',
                     'Low warm','Average warm','Very warm','Extreme warm','Exceptional warm']

# ---------- 3 · Relative Wetness (sm) — 10 bins of 20 % ----------
SM_BOUNDS = [-100, -80, -60, -40, -20, 0, 20, 40, 60, 80, 100]
SM_COLORS = ['#990000','#FF6666','#FFA31A','#FFDB4D','#FFF3E6',
             '#E6F2FF','#99CCFF','#00B8E6','#0086B3','#005266']
SM_CATEGORY_LABELS = ['Very dry','Medium dryness','Low dryness',
                      'Low wetness','Medium wetness','Very wet']

# ---------- 4 · Runoff anomaly (ro) — 20 bins of 5 mm ----------
RO_BOUNDS = [-1e9, -45, -40, -35, -30, -25, -20, -15, -10, -5,
                  0,   5,  10,  15,  20,  25,  30,  35,  40, 45, 1e9]
RO_COLORS = [
    '#660000','#A50000','#C00000','#E11400','#FF3200','#FF6000','#FFA000',
    '#FFC03C','#FFE878','#FFFADC','#E6FFFF','#C8FAFF','#A0F0FF','#82D2FF',
    '#50B4FA','#3CA0F0','#1E6EC8','#0F4BA5','#0A3278','#000066',
]
RO_CATEGORY_LABELS = ['Extreme deficit','Very high deficit','High deficit','Medium deficit',
                      'Less deficit','Less surplus','Medium surplus','High surplus',
                      'Very high surplus','Extreme surplus']

# ---------- 5 · Evapotranspiration anomaly (ET) — 18 bins of 2.5 mm ----------
ET_BOUNDS = [-1e9, -20, -17.5, -15, -12.5, -10, -7.5, -5, -2.5,
                 0,   2.5,   5,   7.5,  10,  12.5, 15,  17.5, 20, 1e9]
ET_COLORS = ['#A50000','#C00000','#FF1400','#FF3200','#FF6000','#FFA000',
             '#FFC03C','#FFE878','#FFFADC',
             '#E6FFFF','#C8FAFF','#A0F0FF','#82D2FF','#50B4FA',
             '#3CA0F0','#1E6EC8','#0F4BA5','#0A3278']
ET_CATEGORY_LABELS = ['Extreme low','Very low','Low','Below average',
                      'Above average','High','Very high','Extreme high']

# ---------- 6 · Streamflow Network (Q) — 20 bins of 5 percentile (same as P) ----------
Q_BOUNDS = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
Q_COLORS = list(P_COLORS)
Q_CATEGORY_LABELS = list(P_CATEGORY_LABELS)

# ---------- 7 · Station Streamflow (Station_Q) — 11 unequal bins ----------
SQ_BOUNDS = [0, 2, 5, 10, 20, 30, 70, 80, 90, 95, 98, 100]
SQ_COLORS = ['#4D2600','#993300','#FF0000','#FF9900','#FFFF00',
             '#FFFFFF','#BFFF00','#99CC00','#33CC33','#009900','#3377FF']
SQ_CATEGORY_LABELS = ['Exceptional low','Extreme low','Very low','Low','Below average',
                      'Average','Above average','High','Very high','Extreme high','Exceptional high']

# ---------------------------------------------------------------------------
# Per-parameter configuration table
# ---------------------------------------------------------------------------
PARAMS = {
    'Rainfall': {
        'file_prefix' : 'P',
        'kind'        : 'grid',
        'bounds'      : P_BOUNDS,
        'colors'      : P_COLORS,
        'category_lbl': P_CATEGORY_LABELS,
        'cb_title'    : 'Rainfall\nPercentile',
        'description' : 'Monthly rainfall expressed as a percentile of the\n'
                        'historical record. Low percentiles (red) indicate\n'
                        'drier-than-normal conditions, high percentiles\n'
                        '(blue) indicate wetter-than-normal conditions.',
        'lowest_label': 'August (2002)',
        'lowest_tag'  : 'Driest',
        'lowest_color': '#C40000',
        'highest_label':'December (1997)',
        'highest_tag' : 'Wettest',
        'highest_color':'#0033A0',
        'tick_format' : '{:.0f}',
    },
    'Temperature': {
        'file_prefix' : 'T',
        'kind'        : 'grid',
        'bounds'      : T_BOUNDS,
        'colors'      : T_COLORS,
        'category_lbl': T_CATEGORY_LABELS,
        'cb_title'    : 'Temperature\nAnomaly (°C)',
        'description' : 'Surface air temperature anomaly compared to the\n'
                        'long-term mean for the same month. Negative\n'
                        'values (blue) indicate cooler-than-average\n'
                        'conditions; positive values (red) indicate\n'
                        'warmer-than-average conditions.',
        'lowest_label': 'May (2023)',
        'lowest_tag'  : 'Coldest month',
        'lowest_color': '#0033A0',     # blue — cold
        'highest_label':'April (2010)',
        'highest_tag' : 'Warmest month',
        'highest_color':'#C40000',     # red  — warm
        'tick_format' : '{:.1f}',
    },
    'Relative_Wetness': {
        'file_prefix' : 'sm',
        'kind'        : 'grid',
        'bounds'      : SM_BOUNDS,
        'colors'      : SM_COLORS,
        'category_lbl': SM_CATEGORY_LABELS,
        'cb_title'    : 'Relative Wetness (%)',
        'description' : 'Soil-moisture anomaly (60 cm depth) as a % of\n'
                        'maximum (positive wetness) or minimum (negative\n'
                        'wetness) soil moisture anomaly.',
        'lowest_label': 'August (2002)',
        'lowest_tag'  : 'Driest',
        'lowest_color': '#C40000',
        'highest_label':'February (2022)',
        'highest_tag' : 'Wettest',
        'highest_color':'#0033A0',
        'tick_format' : '{:.0f}',
    },
    'Total_Runoff': {
        'file_prefix' : 'ro',
        'kind'        : 'grid',
        'bounds'      : RO_BOUNDS,
        'colors'      : RO_COLORS,
        'category_lbl': RO_CATEGORY_LABELS,
        'cb_title'    : 'Total Runoff\nAnomaly (mm)',
        'description' : 'Total runoff anomaly compared to the long-term\n'
                        'monthly mean. Red (negative) = deficit, blue\n'
                        '(positive) = surplus runoff.',
        'lowest_label': 'August (2002)',
        'lowest_tag'  : 'Lowest',
        'lowest_color': '#C40000',
        'highest_label':'October (2019)',
        'highest_tag' : 'Highest',
        'highest_color':'#0033A0',
        'tick_format' : '{:.0f}',
    },
    'Evapotranspiration': {
        'file_prefix' : 'ET',
        'kind'        : 'grid',
        'bounds'      : ET_BOUNDS,
        'colors'      : ET_COLORS,
        'category_lbl': ET_CATEGORY_LABELS,
        'cb_title'    : 'Evapotranspiration\nAnomaly (mm)',
        'description' : 'Evapotranspiration anomaly compared to the\n'
                        'long-term monthly mean. Red = below normal,\n'
                        'blue = above normal.',
        'lowest_label': 'July (2002)',
        'lowest_tag'  : 'Lowest',
        'lowest_color': '#C40000',
        'highest_label':'June (2021)',
        'highest_tag' : 'Highest',
        'highest_color':'#0033A0',
        'tick_format' : '{:.1f}',
    },
    'Streamflow_Network': {
        'file_prefix' : 'Q',
        'kind'        : 'network',
        'bounds'      : Q_BOUNDS,
        'colors'      : Q_COLORS,
        'category_lbl': Q_CATEGORY_LABELS,
        'cb_title'    : 'Streamflow\nPercentile',
        'description' : 'Streamflow percentile across India\'s stream\n'
                        'network, plotted along river segments.',
        'lowest_label': 'August (2002)',
        'lowest_tag'  : 'Lowest flow',
        'lowest_color': '#C40000',
        'highest_label':'January (2022)',
        'highest_tag' : 'Highest flow',
        'highest_color':'#0033A0',
        'tick_format' : '{:.0f}',
    },
    'Streamflow_Stations': {
        'file_prefix' : 'Station_Q',
        'kind'        : 'station',
        'bounds'      : SQ_BOUNDS,
        'colors'      : SQ_COLORS,
        'category_lbl': SQ_CATEGORY_LABELS,
        'cb_title'    : 'Streamflow\nPercentile',
        'description' : 'Monthly streamflow percentile observed at gauge\n'
                        'stations across India.',
        'lowest_label': 'June (2003)',
        'lowest_tag'  : 'Lowest flow',
        'lowest_color': '#C40000',
        'highest_label':'June (2025)',
        'highest_tag' : 'Highest flow',
        'highest_color':'#0033A0',
        'tick_format' : '{:.0f}',
    },
}

# Column-name mapping: every input file has the same 11 columns.
COLUMN_KEYS = ['c1','c2','current','forecast','prev1','prev2','prev3','prev4',
               'last_year','lowest','highest']
PLOT_COLUMNS = COLUMN_KEYS[2:]    # the 9 value columns we actually plot

# Map column-key → (panel_name, dashboard slot)
PANEL_NAMES = {
    'current'  : 'Current',
    'forecast' : 'Forecast',
    'prev1'    : 'Previous1',
    'prev2'    : 'Previous2',
    'prev3'    : 'Previous3',
    'prev4'    : 'Previous4',
    'last_year': 'LastYear',
    'lowest'   : 'Lowest',
    'highest'  : 'Highest',
}

print(f"✓ {len(PARAMS)} parameters configured")


# Column-key list and panel-label mapping (used by data loader and dashboard composer)
COLUMN_KEYS = ['c1','c2','current','forecast','prev1','prev2','prev3','prev4',
               'last_year','lowest','highest']
PLOT_COLUMNS = COLUMN_KEYS[2:]
PANEL_NAMES = {'current':'Current','forecast':'Forecast','prev1':'Previous1',
               'prev2':'Previous2','prev3':'Previous3','prev4':'Previous4',
               'last_year':'LastYear','lowest':'Lowest','highest':'Highest'}

print(f'✓ {len(PARAMS)} parameters configured')


## 4 · Data loading + panel renderers + legend renderer

The next three code cells are **lifted verbatim from `IHO_Maps_Dashboard.ipynb`** so the published dashboards on indiahydrolook.in are reproduced exactly. This includes:

- `load_param(prefix)` — reads one of the 7 input files, auto-detects the `(lon, lat)` swap in `Station_Q_*`.
- `render_grid` / `render_network` / `render_station` — the three layouts used by the parameter dashboards (smooth interpolated grid, stream-network scatter, gauge-station scatter with river basemap).
- `draw_legend` — the central colorbar with vertical category labels above and vertical tick labels below, plus the title and italic description on the right.

Nothing changes here from the original maps notebook; this preserves dashboard fidelity.

In [ ]:
def load_param(prefix):
    '''Read one of the 7 input files into a DataFrame with proper lat/lon columns.'''
    path = os.path.join(INPUT_DIR, f'{prefix}_{FORECAST_DATE_STR}')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Cannot find {path}')
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLUMN_KEYS, engine='python')

    # Auto-detect lat/lon order: India latitudes are ≤ 38°, longitudes ≥ 65°.
    if df['c1'].max() > 50:                # c1 is longitude
        df = df.rename(columns={'c1': 'lon', 'c2': 'lat'})
    else:
        df = df.rename(columns={'c1': 'lat', 'c2': 'lon'})

    return df


def points_to_grid(df, value_col, resolution=0.25):
    '''Reconstruct an (n_lat × n_lon) NaN-padded grid for imshow/pcolormesh.'''
    lat_min = np.floor(df['lat'].min()/resolution)*resolution
    lat_max = np.ceil (df['lat'].max()/resolution)*resolution
    lon_min = np.floor(df['lon'].min()/resolution)*resolution
    lon_max = np.ceil (df['lon'].max()/resolution)*resolution

    n_lat = int(round((lat_max - lat_min)/resolution)) + 1
    n_lon = int(round((lon_max - lon_min)/resolution)) + 1

    grid = np.full((n_lat, n_lon), np.nan, dtype=np.float32)
    i = np.round((df['lat'].to_numpy() - lat_min)/resolution).astype(int)
    j = np.round((df['lon'].to_numpy() - lon_min)/resolution).astype(int)
    valid = (i >= 0) & (i < n_lat) & (j >= 0) & (j < n_lon)
    grid[i[valid], j[valid]] = df[value_col].to_numpy()[valid]

    return grid, lat_min, lat_max, lon_min, lon_max


# Sanity check
sample = load_param('P')
print('Rainfall data:', sample.shape, '— lat range:', sample.lat.min(), '..', sample.lat.max())


# Sanity peek
_sample = load_param("P")
print(f"Sample (Rainfall): {_sample.shape}  lat={_sample.lat.min():.2f}–{_sample.lat.max():.2f}  lon={_sample.lon.min():.2f}–{_sample.lon.max():.2f}")


In [ ]:
# ============================================================
# Panel renderers — lifted verbatim from IHO_Maps_Dashboard.ipynb
# (so dashboard fidelity matches the published indiahydrolook.in PDFs)
# ============================================================

# Bounding box for India incl. PoK + Aksai Chin (used for axis limits)
INDIA_BBOX = (66.0, 6.5, 98.5, 38.5)        # (lon_min, lat_min, lon_max, lat_max)

# Resolution of the fine grid produced by interpolation. Lower = smoother
# but slower. 0.08° gives ~360 × 380 cells over India and renders in ~0.5 s.
FINE_RESOLUTION = 0.08

from scipy.interpolate import griddata
from shapely import vectorized as shp_vec

# ---------------------------------------------------------------------------
# Streamflow-at-gauge-stations basemap
# ---------------------------------------------------------------------------
# The cleaned major-river basemap is loaded from your Drive folder.
# It is used ONLY for the 'Streamflow at Gauge Stations' maps/dashboard.
RIVER_BASEMAP_PATHS = [
    '/content/drive/MyDrive/Hydrologic_Outlook/WRIS_IndianRivers.geojsonl',
    '/content/drive/MyDrive/Hydrologic_Outlook/WRIS_IndianRivers.geojson',
    '/content/drive/MyDrive/Hydrologic_Outlook/WRIS_IndianRivers.shp',
    '/mnt/data/WRIS_IndianRivers.geojsonl',   # fallback for local testing
]

_RIVER_BASEMAP_GDF = None

def make_cmap_norm(colors, bounds):
    cmap = ListedColormap(colors)
    cmap.set_bad('white', alpha=0.0)
    norm = BoundaryNorm(bounds, cmap.N)
    return cmap, norm


def add_basemap(ax, lw_state=0.65, lw_dist=0.18):
    '''Draw district boundaries (thin) then state boundaries (thicker, on top).'''
    if districts_gdf is not None:
        districts_gdf.boundary.plot(ax=ax, linewidth=lw_dist, color='#444444',
                                    alpha=0.6, zorder=5)
    states_gdf.boundary.plot(ax=ax, linewidth=lw_state, color='black', zorder=6)


def _resolve_river_basemap_path():
    for p in RIVER_BASEMAP_PATHS:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(
        "Could not find WRIS_IndianRivers.geojsonl in the expected Drive path. "
        "Please place it in /content/drive/MyDrive/Hydrologic_Outlook/."
    )


def get_river_basemap_gdf():
    '''Lazy-load the cleaned major-river GeoJSONL used as a basemap for station panels.'''
    global _RIVER_BASEMAP_GDF
    if _RIVER_BASEMAP_GDF is None:
        river_path = _resolve_river_basemap_path()
        _RIVER_BASEMAP_GDF = gpd.read_file(river_path)

        # Keep only valid geometries and, if present, drop anything tagged as minor.
        _RIVER_BASEMAP_GDF = _RIVER_BASEMAP_GDF[_RIVER_BASEMAP_GDF.geometry.notna()].copy()
        if 'layer' in _RIVER_BASEMAP_GDF.columns:
            _RIVER_BASEMAP_GDF = _RIVER_BASEMAP_GDF[
                ~_RIVER_BASEMAP_GDF['layer'].astype(str).str.contains('minor', case=False, na=False)
            ].copy()

        if _RIVER_BASEMAP_GDF.crs is None:
            _RIVER_BASEMAP_GDF = _RIVER_BASEMAP_GDF.set_crs(epsg=4326)
        else:
            _RIVER_BASEMAP_GDF = _RIVER_BASEMAP_GDF.to_crs(4326)

        print(f"✓ River basemap loaded: {len(_RIVER_BASEMAP_GDF)} features from {river_path}")

    return _RIVER_BASEMAP_GDF


def add_river_basemap(ax, color='#4C6DFF', linewidth=0.55, alpha=0.95):
    '''Major-river line basemap used only for gauge-station panels.'''
    river_gdf = get_river_basemap_gdf()
    river_gdf.plot(ax=ax, color=color, linewidth=linewidth, alpha=alpha, zorder=0)


def render_grid(ax, df, value_col, cmap, norm):
    '''Smooth gridded rendering: interpolate sparse 0.25° points to a fine grid,
    then mask everything outside India's official boundary so the fill stops
    exactly at the border.'''
    lons = df['lon'].to_numpy()
    lats = df['lat'].to_numpy()
    vals = df[value_col].to_numpy()

    pad = 0.5
    lo_min, lo_max = lons.min() - pad, lons.max() + pad
    la_min, la_max = lats.min() - pad, lats.max() + pad

    fine_lons = np.arange(lo_min, lo_max + FINE_RESOLUTION, FINE_RESOLUTION)
    fine_lats = np.arange(la_min, la_max + FINE_RESOLUTION, FINE_RESOLUTION)
    LX, LY = np.meshgrid(fine_lons, fine_lats)

    pts = np.column_stack([lons, lats])
    fine_lin = griddata(pts, vals, (LX, LY), method='linear')
    fine_nn  = griddata(pts, vals, (LX, LY), method='nearest')
    fine_grid = np.where(np.isnan(fine_lin), fine_nn, fine_lin)

    # Clip to India's official boundary (incl. PoK + Aksai Chin)
    inside = shp_vec.contains(INDIA_UNION, LX, LY)
    fine_grid = np.where(inside, fine_grid, np.nan)

    masked = np.ma.array(fine_grid, mask=np.isnan(fine_grid))
    ax.pcolormesh(LX, LY, masked, cmap=cmap, norm=norm,
                  shading='nearest', antialiased=False, zorder=1)

    add_basemap(ax)
    ax.set_xlim(INDIA_BBOX[0], INDIA_BBOX[2])
    ax.set_ylim(INDIA_BBOX[1], INDIA_BBOX[3])
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)


def render_network(ax, df, value_col, cmap, norm, marker_size=0.6):
    '''Stream network: many tiny coloured markers approximating a river line.'''
    add_basemap(ax)
    ax.scatter(df['lon'], df['lat'], c=df[value_col], cmap=cmap, norm=norm,
               s=marker_size, marker='o', linewidths=0, zorder=2)
    ax.set_xlim(INDIA_BBOX[0], INDIA_BBOX[2])
    ax.set_ylim(INDIA_BBOX[1], INDIA_BBOX[3])
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)


def render_station(ax, df, value_col, cmap, norm, marker_size=20):
    '''Gauge stations: medium circles with thin black edge plus major-river basemap.'''
    add_river_basemap(ax)
    add_basemap(ax)
    ax.scatter(df['lon'], df['lat'], c=df[value_col], cmap=cmap, norm=norm,
               s=marker_size, marker='o', edgecolors='black',
               linewidths=0.3, zorder=3)
    ax.set_xlim(INDIA_BBOX[0], INDIA_BBOX[2])
    ax.set_ylim(INDIA_BBOX[1], INDIA_BBOX[3])
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)


def render_panel(ax, df, value_col, cfg):
    cmap, norm = make_cmap_norm(cfg['colors'], cfg['bounds'])
    if   cfg['kind'] == 'grid'   : render_grid   (ax, df, value_col, cmap, norm)
    elif cfg['kind'] == 'network': render_network(ax, df, value_col, cmap, norm)
    elif cfg['kind'] == 'station': render_station(ax, df, value_col, cmap, norm)
    else: raise ValueError(cfg['kind'])

print("✓ panel renderers ready (verbatim from IHO_Maps_Dashboard notebook)")


In [ ]:
# Legend renderer (verbatim from IHO_Maps_Dashboard notebook)

def draw_legend(fig, ax_pos, cfg):
    '''Render the dashboard's central legend (colorbar + category labels + title).

    Layout (matches the reference dashboards on indiahydrolook.in):
        ┌───────────────────────────────────────────────────────────┐
        │ [category labels above colorbar — vertical text]          │
        │ ████████████████████████████████████      Title           │
        │ ──────────────────────────────────        (description    │
        │ [tick labels below colorbar — vertical]    underneath)    │
        └───────────────────────────────────────────────────────────┘
    '''
    L, B, W, H = ax_pos
    bounds  = cfg['bounds']
    colors  = cfg['colors']
    cats    = cfg['category_lbl']
    fmt     = cfg['tick_format']

    # ----- colorbar (about half the legend width, left-leaning)
    cb_h = 0.034
    cb_w = W * 0.50
    cb_l = L + W * 0.04
    cb_b = B + H * 0.42

    cax = fig.add_axes([cb_l, cb_b, cb_w, cb_h])
    cmap, norm = make_cmap_norm(colors, bounds)
    cb = mpl.colorbar.ColorbarBase(cax, cmap=cmap, norm=norm,
                                   orientation='horizontal',
                                   spacing='uniform',
                                   ticks=[])
    cb.outline.set_linewidth(0.4)

    # ----- bin-edge tick labels below the bar (vertical so they don't collide)
    n = len(colors)
    for i, edge in enumerate(bounds):
        x = i / n
        if abs(edge) >= 1e8:
            label = f'< {fmt.format(bounds[1])}' if i == 0 else f'> {fmt.format(bounds[-2])}'
        elif i == 0 and bounds[0] == 0 and bounds[-1] == 100:
            label = '0'
        else:
            label = fmt.format(edge)
        cax.text(x, -0.7, label, transform=cax.transAxes,
                 ha='center', va='top', fontsize=7.5, rotation=90)

    # ----- category labels above the bar
    n_cats = len(cats)
    for k, txt in enumerate(cats):
        x = (k + 0.5) / n_cats
        cax.text(x, 1.6, txt, transform=cax.transAxes,
                 ha='center', va='bottom', fontsize=8.5,
                 rotation=90, fontweight='bold')

    # ----- title and description on the right side of the colorbar
    text_x = cb_l + cb_w + W * 0.05
    fig.text(text_x, cb_b + cb_h * 1.0, cfg['cb_title'],
             ha='left', va='top', fontsize=15, fontweight='bold',
             linespacing=1.1)
    fig.text(text_x, cb_b - cb_h * 0.6, cfg['description'],
             ha='left', va='top', fontsize=8.0, color='#222',
             style='italic', linespacing=1.4)


## 5 · Render the individual map PNGs (verbatim)

Verbatim `save_individual_map` + master loop from `IHO_Maps_Dashboard.ipynb`. Writes 7 parameters × 9 columns = 63 PNGs into `Output/All_Maps/<Parameter>/`.

In [ ]:
def save_individual_map(df, value_col, cfg, out_dir, panel_label):
    # Figure is taller so the dashboard-style legend has room beneath the map.
    fig = plt.figure(figsize=(10, 12), dpi=150, facecolor='white')

    # Map occupies the upper ~72 % of the figure.
    ax = fig.add_axes([0.04, 0.30, 0.92, 0.66])
    render_panel(ax, df, value_col, cfg)
    ax.set_title(panel_label, fontsize=16, fontweight='bold', pad=10)

    # Full legend (same one used on the dashboard) drawn in the lower band.
    # draw_legend takes a figure-relative (L, B, W, H) box and renders:
    #   * vertical category labels above the colorbar
    #   * the colorbar itself
    #   * vertical tick-value labels below
    #   * title + italic description on the right
    draw_legend(fig, (0.04, 0.02, 0.92, 0.22), cfg)

    out = Path(out_dir) / f'{panel_label}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out


def panel_label_for(col_key, cfg):
    '''Human-readable label, e.g. 'February (Current)'.'''
    L = LABELS
    if col_key == 'current'  : return f"{L['current']} (Current)"
    if col_key == 'forecast' : return f"{L['forecast']} (Forecast)"
    if col_key == 'prev1'    : return L['prev1']
    if col_key == 'prev2'    : return L['prev2']
    if col_key == 'prev3'    : return L['prev3']
    if col_key == 'prev4'    : return L['prev4']
    if col_key == 'last_year': return L['last_year']
    if col_key == 'lowest'   : return cfg['lowest_label']
    if col_key == 'highest'  : return cfg['highest_label']
    return col_key


# ---- master loop -----------------------------------------------------------
print("Generating individual maps …\n")
for pretty_name, cfg in PARAMS.items():
    pdir = Path(ALL_MAPS_DIR) / pretty_name
    pdir.mkdir(parents=True, exist_ok=True)
    print(f"▸ {pretty_name}")
    df = load_param(cfg['file_prefix'])
    for col in PLOT_COLUMNS:
        label = panel_label_for(col, cfg)
        out   = save_individual_map(df, col, cfg, pdir, label)
        print(f"   • {out.name}")
print("\n✓ All individual maps written to", ALL_MAPS_DIR)


## 6 · Compose the 9-panel dashboards (verbatim)

Verbatim `LAYOUT_BOXES`, `build_dashboard`, and master loop from `IHO_Maps_Dashboard.ipynb`. Each dashboard PNG has a large current-month panel on the upper left, four previous-month panels in a row, the central legend, the large forecast panel on the lower left, and last_year + lowest + highest panels (the latter two framed in red and blue with 'Driest'/'Wettest'/'Lowest'/'Highest' badges).

In [ ]:
# box_2d → (left, bottom, width, height) in figure coords (origin bottom-left)
def _box2axes(box):
    y1, x1, y2, x2 = box
    return (x1/1000, 1 - y2/1000, (x2-x1)/1000, (y2-y1)/1000)

LAYOUT_BOXES = {
    'current'  : [ 39,  11, 473, 298],
    'prev1'    : [ 49, 319, 296, 485],
    'prev2'    : [ 49, 489, 296, 654],
    'prev3'    : [ 49, 660, 296, 827],
    'prev4'    : [ 49, 831, 296, 997],
    'legend'   : [334, 224, 532, 814],
    'forecast' : [538,  11, 971, 289],
    'last_year': [640, 295, 971, 510],
    'lowest'   : [640, 517, 971, 735],
    'highest'  : [640, 765, 971, 983],
}
LAYOUT = {k: _box2axes(v) for k, v in LAYOUT_BOXES.items()}


def build_dashboard(pretty_name, cfg, out_path):
    fig = plt.figure(figsize=(22, 15.4), dpi=140, facecolor='white')

    df = load_param(cfg['file_prefix'])

    # Pairs (column-key, dashboard-slot, dashboard-label, font-weight, extra-tag, frame-color)
    panels = [
        ('current',  'current',  panel_label_for('current',  cfg), 'bold', None, None),
        ('prev1',    'prev1',    panel_label_for('prev1',    cfg), 'bold', None, None),
        ('prev2',    'prev2',    panel_label_for('prev2',    cfg), 'bold', None, None),
        ('prev3',    'prev3',    panel_label_for('prev3',    cfg), 'bold', None, None),
        ('prev4',    'prev4',    panel_label_for('prev4',    cfg), 'bold', None, None),
        ('forecast', 'forecast', panel_label_for('forecast', cfg), 'bold', None, None),
        ('last_year','last_year',panel_label_for('last_year',cfg), 'bold', None, None),
        ('lowest',   'lowest',   cfg['lowest_label'],              'bold', cfg['lowest_tag'],  cfg['lowest_color']),
        ('highest',  'highest',  cfg['highest_label'],             'bold', cfg['highest_tag'], cfg['highest_color']),
    ]

    for col_key, slot, label, weight, tag, tag_color in panels:
        L, B, W, H = LAYOUT[slot]
        ax = fig.add_axes([L, B, W, H])
        render_panel(ax, df, col_key, cfg)
        # Title above each map
        ax.text(0.5, 1.02, label, transform=ax.transAxes,
                ha='center', va='bottom',
                fontsize=15 if slot in ('current','forecast') else 12,
                fontweight=weight)
        # 'Driest' / 'Wettest' / 'Lowest' / 'Highest' tag in the corner of extreme panels
        if tag:
            ax.text(0.97, 0.97, tag, transform=ax.transAxes,
                    ha='right', va='top', fontsize=11, fontweight='bold',
                    color=tag_color,
                    bbox=dict(boxstyle='round,pad=0.25',
                              facecolor='white', edgecolor=tag_color, linewidth=1.2))
            # red/blue frame around extreme panels
            for s in ax.spines.values(): s.set_visible(False)
            rect = mpatches.Rectangle((0,0), 1, 1, transform=ax.transAxes,
                                      fill=False, lw=2.5, edgecolor=tag_color, clip_on=False)
            ax.add_patch(rect)

    # ---- centre legend ------------------------------------------------------
    draw_legend(fig, LAYOUT['legend'], cfg)

    # ---- save ---------------------------------------------------------------
    fig.savefig(out_path, dpi=140, bbox_inches=None, facecolor='white')
    plt.close(fig)


print("Generating dashboards …\n")
for pretty_name, cfg in PARAMS.items():
    out = Path(DASHBOARDS_DIR) / f'{pretty_name}_dashboard.png'
    print(f"▸ {pretty_name}  →  {out.name}")
    build_dashboard(pretty_name, cfg, out)
print("\n✓ All dashboards written to", DASHBOARDS_DIR)


## 7 · Per-region category histograms

Instead of averaging each region's values into a single number and binning it (which loses geographic nuance — a region with mixed wet and dry cells gets called "near-normal", and a region with a few extreme cells gets called "exceptionally low"), we compute the **distribution** of values across each region.

For every (parameter, region, time) combination we record:
- `pct_below` / `pct_near` / `pct_above` — what fraction of cells fall into each directional band
- `pct_far_below` / `pct_far_above` — what fraction sit in the tails
- a narrative-style descriptor (`"higher-than-normal rainfall"`, `"a high runoff deficit"`, `"high relative wetness"`) chosen by simple rules

This is the **key data-quality fix**: the LLM never gets a single ambiguous "mean" — it sees the actual shape of the regional distribution and a descriptor that matches the dashboard's vocabulary.

In [ ]:
"""
========================================================================
Section 7 · Per-region category histograms (the key data-quality fix)
========================================================================
Instead of averaging each region's values into a single number and then binning
that average, we compute the *distribution* of values across each region — what
fraction of cells fall into the dashboard's directional bands.

A region whose cells are 70% below-normal and 5% above-normal gets called
'below-normal'. A region whose cells are 35% below / 50% near / 15% above gets
called 'near-normal with a dry skew' — capturing the actual geographic story.

This fixes the previous failure mode where a region with mixed wet/dry cells
got mean ≈ 0 and was labelled 'near-normal', losing the real signal. It also
fixes the opposite failure where one extreme cell pulled the mean into a tail
band and the LLM was told the whole region was 'exceptionally low' when most
of it was actually near-normal.
"""

import pandas as pd

# Region masks (unchanged — coarse lat/lon bounding boxes for India)
def region_mask(df, region):
    lat, lon = df['lat'], df['lon']
    if region == 'north':         return (lat >= 28) & lon.between(73, 88)
    if region == 'northeast':     return (lat >= 22) & (lon >= 89)
    if region == 'northwest':     return (lat >= 24) & (lon < 76)
    if region == 'central':       return lat.between(20, 26) & lon.between(74, 86)
    if region == 'east':          return lat.between(20, 27) & lon.between(83, 89)
    if region == 'west':          return lat.between(17, 24) & lon.between(68, 75)
    if region == 'south':         return lat < 18
    if region == 'central_east':  return lat.between(18, 24) & lon.between(80, 86)
    if region == 'all_india':     return pd.Series(True, index=df.index)
    raise ValueError(region)

REGIONS = ['north','northeast','northwest','central','east','west','south','central_east']

# Direction bands per parameter — these align with how the dashboards (and
# the published bulletin's prose) actually talk. Each band is a fraction of cells.
def directional_summary(values, parameter):
    """Return a structured dict describing the distribution: what fraction of
    cells fall below normal, near normal, above normal, plus tail fractions for
    'extreme' qualifiers. The thresholds are chosen to match the bulletin's vocabulary."""
    v = pd.Series(values).dropna().astype(float)
    if len(v) == 0:
        return {'dominant': 'no_data', 'narrative': 'no data', 'n': 0}

    if parameter == 'P' or parameter == 'Q' or parameter == 'Station_Q':
        # Percentile 0–100. "below" = drier than the 35th percentile climatology day.
        below     = (v < 35).mean()
        near      = ((v >= 35) & (v < 65)).mean()
        above     = (v >= 65).mean()
        far_below = (v < 15).mean()
        far_above = (v >= 85).mean()
    elif parameter == 'sm':
        # Relative wetness anomaly, –100 to +100. Negative = dry.
        below     = (v < -20).mean()
        near      = ((v >= -20) & (v <= 20)).mean()
        above     = (v > 20).mean()
        far_below = (v < -60).mean()
        far_above = (v >  60).mean()
    elif parameter == 'T':
        # Temperature anomaly in °C.
        below     = (v < -0.5).mean()
        near      = ((v >= -0.5) & (v <= 0.5)).mean()
        above     = (v >  0.5).mean()
        far_below = (v < -2.0).mean()
        far_above = (v >  2.0).mean()
    elif parameter in ('ro', 'ET'):
        # mm anomaly.
        below     = (v < -5).mean()
        near      = ((v >= -5) & (v <= 5)).mean()
        above     = (v >  5).mean()
        far_below = (v < -20).mean()
        far_above = (v >  20).mean()
    else:
        raise ValueError(parameter)

    fractions = {'below': float(below), 'near': float(near), 'above': float(above)}
    dominant_key = max(fractions, key=fractions.get)
    dominant_frac = fractions[dominant_key]

    # Vocabulary per parameter and direction — picked to match the bulletin's tone
    VOCAB = {
        'P': {'below': 'lower-than-normal rainfall',     'near': 'near-normal rainfall',
              'above': 'higher-than-normal rainfall',
              'far_below': 'very low rainfall',          'far_above': 'very high rainfall'},
        'T': {'below': 'cooler-than-normal temperatures', 'near': 'near-normal temperatures',
              'above': 'warmer-than-normal temperatures',
              'far_below': 'much cooler than normal',    'far_above': 'much warmer than normal'},
        'sm':{'below': 'drier-than-normal soil',          'near': 'near-normal soil moisture',
              'above': 'wetter-than-normal soil',
              'far_below': 'very dry soil',              'far_above': 'high relative wetness'},
        'ro':{'below': 'a runoff deficit',                'near': 'near-normal runoff',
              'above': 'a runoff surplus',
              'far_below': 'a high runoff deficit',      'far_above': 'a high runoff surplus'},
        'ET':{'below': 'reduced evapotranspiration',      'near': 'near-normal evapotranspiration',
              'above': 'elevated evapotranspiration',
              'far_below': 'very low evapotranspiration','far_above': 'very high evapotranspiration'},
        'Q': {'below': 'lower-than-normal streamflow',    'near': 'near-normal streamflow',
              'above': 'higher-than-normal streamflow',
              'far_below': 'very low streamflow',        'far_above': 'very high streamflow'},
        'Station_Q': {'below': 'lower-than-normal streamflow', 'near': 'near-normal streamflow',
              'above': 'higher-than-normal streamflow',
              'far_below': 'very low streamflow',        'far_above': 'very high streamflow'},
    }
    vocab = VOCAB[parameter]

    # Choose the narrative descriptor:
    #   - If a tail captures >25 % of cells, use the 'far_*' wording
    #   - Otherwise if the dominant direction has >55 % of cells, use it neatly
    #   - Otherwise call it 'mixed' with the dominant direction
    if dominant_key == 'below' and far_below > 0.25:
        narrative = vocab['far_below']
    elif dominant_key == 'above' and far_above > 0.25:
        narrative = vocab['far_above']
    elif dominant_frac > 0.55:
        narrative = vocab[dominant_key]
    else:
        narrative = 'mixed conditions, leaning toward ' + vocab[dominant_key]

    return {
        'dominant':       dominant_key,
        'dominant_frac':  round(dominant_frac, 2),
        'pct_below':      int(round(below     * 100)),
        'pct_near':       int(round(near      * 100)),
        'pct_above':      int(round(above     * 100)),
        'pct_far_below':  int(round(far_below * 100)),
        'pct_far_above':  int(round(far_above * 100)),
        'narrative':      narrative,
        'n':              int(len(v)),
    }


# ----------------------------------------------------------------------
# Compute the evidence packet — one entry per (parameter, region, time).
# This is what the LLM sees in Section 9.
# ----------------------------------------------------------------------
VALUE_TYPES = {'P':'percentile','T':'anomaly','sm':'percentile','ro':'anomaly',
               'ET':'anomaly','Q':'percentile','Station_Q':'percentile'}

EVIDENCE = {}
for prefix in ['P','T','sm','ro','ET','Q','Station_Q']:
    df = load_param(prefix)
    EVIDENCE[prefix] = {}
    for region in REGIONS:
        mask = region_mask(df, region)
        if mask.sum() == 0:
            continue
        EVIDENCE[prefix][region] = {
            'current':  directional_summary(df.loc[mask, 'current'],  prefix),
            'forecast': directional_summary(df.loc[mask, 'forecast'], prefix),
        }

# Quick peek at one parameter so you can see the evidence quality
import json
print('Evidence for Rainfall (P), regions × {current, forecast}:')
for region, t in EVIDENCE['P'].items():
    cur = t['current']
    fc  = t['forecast']
    print(f"  {region:14s} current={cur['narrative']:45s} ({cur['pct_below']}/{cur['pct_near']}/{cur['pct_above']})  "
          f"forecast={fc['narrative']}")


## 8 · Load Qwen3.5-4B and unpack HydroQA

We reuse the `TransformersBackend` from the HydroQA package — it gives us a stable `chat(system, user)` interface we can call with custom prompts.

**Why 4B and not 2B?** The previous (2B) version made semantic errors that smaller-model paragraph generation is prone to: reversing geographic directions, contradicting itself within a paragraph, dumping region-name lists instead of writing prose. 4B handles the few-shot style transfer in Section 9 much more reliably. The latency cost is small — paragraph generation runs once a month.

First run downloads ~8 GB of weights to `/root/.cache/huggingface`; subsequent runs are instant from cache (but Colab wipes the cache between sessions — set `HF_HOME` to a Drive path if you want persistence).

In [ ]:
# Unpack HydroQA so we can import the backend class
import zipfile
PROJECT_DIR = Path('/content/hydroqa')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not HYDROQA_ZIP.exists():
    raise FileNotFoundError(f'Place hydroqa.zip at {HYDROQA_ZIP} before running this cell.')
with zipfile.ZipFile(HYDROQA_ZIP) as zf:
    zf.extractall('/content')

if str(PROJECT_DIR/'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR/'src'))

import torch
from llm import TransformersBackend
import llm as _llm
_llm.MODEL_ID = 'Qwen/Qwen3.5-4B'

dtype = torch.float16 if torch.cuda.is_available() else 'auto'
BACKEND = TransformersBackend(model=_llm.MODEL_ID, dtype=dtype, device_map='auto',
                              enable_thinking=False)
print(f'✓ {_llm.MODEL_ID} loaded.  thinking={BACKEND.enable_thinking}')


## 9 · LLM paragraph generation (with few-shot + consistency scoring + retries)

This is the section that produces the 12 paragraphs the PDF needs. The previous version had three failure modes that are all addressed here:

1. **Region-list dumping** ("north, northeast, northwest, central, east, south, central-east experienced …") — fixed by adding *few-shot reference paragraphs* from the actual published outlook. The model now matches the bulletin's writing style ("majority of central parts", "across India except north and northeast", "above-average soil moisture persisted in central and southern parts") instead of inventing region lists.

2. **Factual contradictions with the dashboards** (e.g. "exceptionally low relative wetness" when the wetness map clearly shows blue/teal across India) — fixed by the new Section 7 evidence packet (narrative-style band descriptors instead of single means) plus a *rule-based consistency checker* that scans each generated paragraph for direction-claim ↔ region pairs and counts contradictions against the evidence. Paragraphs with contradictions get regenerated.

3. **Word-budget overshoots** (page 8 was +46% over the target in the previous run) — fixed by accepting up to 5 attempts and picking the best of them (zero contradictions first, then closest-to-target word count).

The LLM never sees raw numbers — only the qualitative band descriptors and a real reference paragraph to mimic. All the geographic inference happens in the deterministic Python in Section 7.

In [ ]:
"""
========================================================================
Section 9 · LLM paragraph generation (with all v3 fixes)
========================================================================
Twelve paragraphs to write. Each has a target word count from the live February
2026 outlook. We use four mechanisms together to land high-quality, factually
faithful prose:

1. **Few-shot examples baked into the prompt** — actual paragraphs from the
   reference bulletin, so the model learns the style ("majority of central parts",
   "across India except", "above-average soil moisture persisted") rather than
   inventing region-name lists.

2. **Narrative-style evidence** — Section 7 already converted the regional
   distributions into natural-language band descriptors ("higher-than-normal
   rainfall", "a high runoff deficit"). The model just needs to weave them
   together; it never has to interpret raw numbers.

3. **Rule-based consistency scoring** — after generation we scan the prose
   for direction-claim ↔ region pairings and check them against the evidence.
   Each contradiction is a point off the score; we regenerate if the score
   is bad and keep the best of N attempts.

4. **Tight word-budget enforcement** — ±15 % of the target, with retries if
   the LLM drifts. We accept the closest-to-budget attempt only if it's also
   contradiction-free; otherwise we prefer the contradiction-free attempt
   even if it's a bit off-budget.
"""

import re

# -----------------------------------------------------------------------
# Target word counts (measured from the live Feb 2026 reference outlook).
# -----------------------------------------------------------------------
TARGETS = {
    'page1_summary':            151,
    'page1_rainfall_temp':       77,
    'page1_sm_ro_et':            84,
    'page1_rivers':              64,
    'page2_rainfall_yellow':     35,
    'page3_temperature_yellow':  42,
    'page4_wetness_yellow':      34,
    'page5_runoff_yellow':       26,
    'page6_et_yellow':           31,
    'page7_stationq_yellow':     34,
    'page8_networkq_yellow':     28,
}
TOLERANCE        = 0.15   # ±15 % around target (default)
# Per-slot tolerance overrides for paragraphs in tight layout boxes on page 1.
# Page 1 has fixed-height regions, so even a small overshoot causes the summary
# box to overlap the body section. We pin those paragraphs more tightly.
TOLERANCE_BY_SLOT = {
    'page1_summary':       0.07,   # ±7%  — top SUMMARY box overlaps body if it grows
    'page1_rainfall_temp': 0.10,   # ±10% — body section needs to fit in 8.2"-wide column
    'page1_sm_ro_et':      0.10,   # ±10% — same
    'page1_rivers':        0.10,   # ±10% — same
}
MAX_ATTEMPTS     = 7      # up to 7 sampling rounds per paragraph (was 5)


# -----------------------------------------------------------------------
# Few-shot reference paragraphs (taken directly from the published Feb 2026
# outlook). The {current_month} / {forecast_month} placeholders are replaced
# with the live values so the examples stay current-month-appropriate.
# -----------------------------------------------------------------------
REFERENCE_EXAMPLES = {
    'page1_summary': (
        "In February, India experienced lower-than-normal rainfall in majority of central "
        "parts, and lower-than-normal rainfall in northeast parts, along with lower-than-"
        "average temperatures and high relative wetness across India, and decreased total "
        "runoff in north-central and north-east India. Above-average soil moisture persisted "
        "in central and southern parts, while runoff deficit affected the north-central and "
        "north-east India. Streamflow in rivers and reaches was also higher across country "
        "except north and northeast India. In March, relatively high rainfall in west, "
        "central, and northeast regions, while low rainfall in west central India is expected. "
        "India is likely to experience relatively higher temperature in central regions, with "
        "relatively very low temperature in northern India and northeast regions. Decreased "
        "ET across country, and high deficit of total runoff in north and northeastern India "
        "are predicted. Moreover, lower streamflow in river-reaches is predicted in north and "
        "central region, while higher streamflow in river-reaches is predicted in south India."
    ),
    'page1_rainfall_temp': (
        "In February, the east and the majority of central India experienced lower rainfall, "
        "while northeast India recorded relatively lower rainfall. In March, a relatively "
        "higher rainfall is predicted in west-central and north India; however, west central "
        "and northeastern parts of India may receive relatively lower rainfall. The temperature "
        "was relatively cooler in February in India, except in the north-east. In March, "
        "above-average temperature is predicted in western India, with potential cold spells "
        "in northern and northeastern parts."
    ),
    'page1_sm_ro_et': (
        "In February, soil moisture was relatively higher than normal in west, central and "
        "south India. In March, above-average soil moisture is going to persist. A very high "
        "surplus in total runoff was observed in central India in February, while in March, "
        "a deficit of total runoff may be recorded in northern India. Evapotranspiration (ET) "
        "in February was reduced in east-central India, which will be reduced in March. "
        "However, a relatively lesser ET in March is expected across India, indicating "
        "decreased moisture loss across country."
    ),
    'page1_rivers': (
        "In February, a relatively very high streamflow in rivers and reaches was observed "
        "in most locations in south India, while a relatively low flow is observed in "
        "northeast parts. In March, a relatively extreme low streamflow is predicted in "
        "south and central regions. Moreover, north-central regions are likely to experience "
        "low streamflow in its river networks, indicating sustained low water volumes in "
        "these areas."
    ),
    'page2_rainfall_yellow': (
        "A relatively lower rainfall in February across India. In March, a higher rainfall "
        "than usual is predicted in northeast and northwest India, while relatively less "
        "rainfall is predicted in eastcentral, and southern parts of country."
    ),
    'page3_temperature_yellow': (
        "In February, a relatively higher temperature than usual was observed across India "
        "except some parts of south, and central-east India. In March, high warm temperature "
        "than usual across India. Moreover, extreme low temperature than usual is predicted "
        "in north and northeast parts."
    ),
    'page4_wetness_yellow': (
        "Soil moisture in February is medium wetter than usual time of year across India "
        "except north and north-east India. In March, low to medium dry wetness (soil "
        "moisture) is persisted in the same regions."
    ),
    'page5_runoff_yellow': (
        "In February, a relatively normal total runoff across country except north-east and "
        "north India. In March, relative lower runoff is predicted in northeast, and north India."
    ),
    'page6_et_yellow': (
        "In February, a less evapotranspiration (ET) than usual in east-central, north, and "
        "northeast region. In March, a relatively low ET is predicted across country except "
        "few parts of central east India."
    ),
    'page7_stationq_yellow': (
        "In February, a relatively higher streamflow for most of the locations except locations "
        "in northeast region. In March, a relatively low streamflow is predicted in central "
        "India, and very low streamflow in north India."
    ),
    'page8_networkq_yellow': (
        "In February, a relatively high flow in the rivers except north-central, and northeast "
        "India. In March, a relatively low flow is predicted in north, central, and south India."
    ),
}


# -----------------------------------------------------------------------
# Consistency checker (rule-based contradiction detector).
# Used to score paragraphs after generation.
# -----------------------------------------------------------------------
PARAM_DIRECTION_PATTERNS = {
    'P': {
        'below': ['low rainfall', 'lower rainfall', 'less rainfall', 'lower-than-normal rainfall',
                  'below-average rainfall', 'below average rainfall', 'reduced rainfall',
                  'rainfall deficit', 'exceptionally low rainfall', 'very low rainfall',
                  'extremely low rainfall', 'extreme low rainfall', 'drier than normal'],
        'above': ['high rainfall', 'higher rainfall', 'more rainfall', 'higher-than-normal rainfall',
                  'above-average rainfall', 'above average rainfall', 'wetter than normal',
                  'very high rainfall', 'extremely high rainfall', 'high precipitation',
                  'higher precipitation', 'very high precipitation', 'increased rainfall'],
    },
    'T': {
        'below': ['cooler temperatures', 'much cooler', 'cooler-than-normal', 'cooler than normal',
                  'cooler than usual', 'colder than normal', 'low temperature', 'lower temperature',
                  'extreme low temperature', 'below-average temperature', 'temperature was cooler',
                  'lower-than-average temperature', 'lower than average temperature'],
        'above': ['warmer temperatures', 'much warmer', 'warmer-than-normal', 'warmer than normal',
                  'warmer than usual', 'higher temperature', 'high temperature', 'hotter',
                  'above-average temperature', 'higher than normal temperature',
                  'high warm temperature', 'higher temperature than usual'],
    },
    'sm': {
        'below': ['drier soil', 'dry soil', 'low wetness', 'lower wetness', 'lower soil moisture',
                  'drier than normal', 'drier-than-normal soil', 'low relative wetness',
                  'lower relative wetness', 'low soil moisture', 'medium dryness', 'very dry',
                  'exceptionally low soil moisture', 'exceptionally low wetness',
                  'exceptionally low relative wetness', 'extremely low soil moisture'],
        'above': ['wetter soil', 'high wetness', 'higher wetness', 'higher soil moisture',
                  'wetter than normal', 'wetter-than-normal soil', 'high relative wetness',
                  'higher relative wetness', 'high soil moisture', 'medium wetness',
                  'very wet', 'medium wetter'],
    },
    'ro': {
        'below': ['runoff deficit', 'lower runoff', 'low runoff', 'reduced runoff', 'less runoff',
                  'high deficit', 'medium deficit', 'extreme deficit', 'low total runoff',
                  'high runoff deficit', 'lower-than-normal runoff'],
        'above': ['runoff surplus', 'higher runoff', 'high runoff', 'more runoff', 'increased runoff',
                  'high surplus', 'medium surplus', 'extreme surplus', 'very high surplus',
                  'higher-than-normal runoff'],
    },
    'ET': {
        'below': ['less evapotranspiration', 'lower evapotranspiration', 'reduced evapotranspiration',
                  'lower et', 'less et', 'low et', 'reduced et', 'low evapotranspiration',
                  'extreme low evapotranspiration', 'very low evapotranspiration',
                  'decreased et', 'decreased evapotranspiration'],
        'above': ['more evapotranspiration', 'higher evapotranspiration', 'increased evapotranspiration',
                  'higher et', 'high et', 'elevated et', 'elevated evapotranspiration',
                  'high evapotranspiration', 'very high evapotranspiration'],
    },
    'Q': {
        'below': ['low streamflow', 'lower streamflow', 'low flow', 'lower flow',
                  'reduced streamflow', 'streamflow deficit', 'very low flow',
                  'extreme low streamflow', 'extreme low flow', 'lower-than-normal streamflow'],
        'above': ['high streamflow', 'higher streamflow', 'high flow', 'higher flow',
                  'increased streamflow', 'streamflow surplus', 'very high flow',
                  'extreme high streamflow', 'higher-than-normal streamflow'],
    },
    'Station_Q': {
        'below': ['low streamflow', 'lower streamflow', 'low flow', 'lower flow',
                  'reduced streamflow', 'streamflow deficit', 'very low flow',
                  'lower-than-normal streamflow'],
        'above': ['high streamflow', 'higher streamflow', 'high flow', 'higher flow',
                  'increased streamflow', 'streamflow surplus', 'very high flow',
                  'higher-than-normal streamflow'],
    },
}

REGION_REGEX = {
    'northeast':    r'\bnorth-?east(?:ern)?\b(?:\s+india)?',
    'northwest':    r'\bnorth-?west(?:ern)?\b(?:\s+india)?',
    'central_east': r'\b(?:central-?east|east-?central)(?:ern)?\b(?:\s+india)?',
    'north':        r'\bnorth(?:ern)?\b(?!-?(?:east|west))(?:\s+india)?',
    'south':        r'\bsouth(?:ern)?\b(?:\s+india)?',
    'east':         r'\beast(?:ern)?\b(?!-?central)(?:\s+india)?',
    'west':         r'\bwest(?:ern)?\b(?:\s+india)?',
    'central':      r'\bcentral\b(?!-?east)(?:\s+india)?',
}

CONTRAST_PATTERNS = [r'\bexcept\b', r'\bexcluding\b', r'\bother\s+than\b',
                     r'\bbut\s+(?:in\s+)?the\b', r'\bnot\s+(?:in|including)\b',
                     r'\bbesides\b', r'\bwhile\b']

def _detect_regions(text_l):
    found = set()
    priority = ['northeast','northwest','central_east','central','east','west','north','south']
    rem = text_l
    for r in priority:
        if re.search(REGION_REGEX[r], rem):
            found.add(r)
            rem = re.sub(REGION_REGEX[r], ' '*9, rem)
    return found

def _detect_excluded(sent_l):
    """Regions following 'except X', 'while Y' etc. are explicitly contrasted —
    they're allowed to have opposite directions to the main claim."""
    excluded = set()
    for pat in CONTRAST_PATTERNS:
        for m in re.finditer(pat, sent_l):
            tail = sent_l[m.end():m.end()+80]
            excluded |= _detect_regions(tail)
    return excluded

def _detect_param_dirs(text_l):
    out = {}
    for param, dirs in PARAM_DIRECTION_PATTERNS.items():
        out[param] = {}
        for d, patterns in dirs.items():
            if any(p in text_l for p in patterns):
                out[param][d] = True
    return out

def score_paragraph(text, evidence, time_key='current'):
    """Return (n_contradictions, details). 0 contradictions = good."""
    issues = []
    text_l = text.lower()
    sentences = re.split(r'(?<=[.;])\s+', text_l)
    for sent in sentences:
        all_regions = _detect_regions(sent)
        if not all_regions: continue
        excluded = _detect_excluded(sent)
        regions = all_regions - excluded
        if not regions: continue
        param_dirs = _detect_param_dirs(sent)
        for param, dirs in param_dirs.items():
            if not dirs: continue
            if 'below' in dirs and 'above' not in dirs: claimed = 'below'
            elif 'above' in dirs and 'below' not in dirs: claimed = 'above'
            else: continue
            for region in regions:
                if region not in evidence.get(param, {}): continue
                actual = evidence[param][region][time_key]['dominant']
                if {claimed, actual} == {'below', 'above'}:
                    issues.append((param, region, claimed, actual))
    return len(issues), issues


# -----------------------------------------------------------------------
# System prompt — keeps Qwen on rails.
# -----------------------------------------------------------------------
SYSTEM_PROMPT = """You are the writing assistant for the India Hydrological Outlook, a monthly PDF published by the Water & Climate Lab at IIT Gandhinagar. Your job is to write ONE paragraph of clean, factual prose for the published outlook.

Strict rules — violating any of these is a failure:

1. **Stick to the evidence.** Use ONLY the regional patterns described in the EVIDENCE block. Do NOT invent regions, months, or numerical values. If the evidence says a region is "higher-than-normal", you cannot write that it's "lower-than-normal" or vice-versa.

2. **Use the bulletin's writing style.** Look at the REFERENCE PARAGRAPH (a real paragraph from a previous month's outlook). Match its tone, sentence structure, and geographic phrasing. Use natural phrases like "majority of central parts", "across India except north and northeast", "above-average soil moisture persisted in central and southern parts" — NOT dry region-name lists like "north, northeast, northwest, central, east, west experienced...".

3. **Do not list all the region names.** That reads like a database dump, not a bulletin. Group regions into geographic clusters with natural prose ("northern India", "central and southern parts", "the northeast", "across India except the south").

4. **Mention both the current month (observed) and the forecast month (predicted).** Use present/past tense for the current month and future tense for the forecast.

5. **Write to the requested WORD COUNT.** Going more than 15% over or under is a failure.

6. **No bullet points, no headers, no Markdown.** One flowing paragraph of prose.

7. **Do NOT wrap the paragraph in quotes, code blocks, or labels.** Output ONLY the paragraph text.

8. **Tone: matter-of-fact, present-tense for the current month, future-tense for the forecast.** Match the style of an official hydrology bulletin — no hype, no marketing language."""


def _evidence_block(params, current_label, forecast_label):
    """Format the regional histogram evidence in a way that's easy for the LLM to consume."""
    PARAM_LABEL = {
        'P':'Rainfall (percentile)', 'T':'Surface air temperature (°C anomaly)',
        'sm':'Relative wetness / soil moisture (% anomaly)',
        'ro':'Total runoff (mm anomaly)', 'ET':'Evapotranspiration (mm anomaly)',
        'Q':'Streamflow at stream network (percentile)',
        'Station_Q':'Streamflow at gauge stations (percentile)',
    }
    lines = []
    for p in params:
        ev = EVIDENCE[p]
        lines.append(f"\n== {PARAM_LABEL[p]} ==")
        for region in REGIONS:
            if region not in ev: continue
            cur = ev[region]['current']
            fc  = ev[region]['forecast']
            r_pretty = region.replace('_', '-')
            lines.append(f"  {r_pretty:14s}  {current_label}: {cur['narrative']}")
            lines.append(f"  {r_pretty:14s}  {forecast_label}: {fc['narrative']}")
    return '\n'.join(lines)


def word_count(s):
    return len(s.split())

def _strip_wrapping(text):
    text = text.strip()
    text = re.sub(r'^```[a-zA-Z]*\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    text = re.sub(r'^(?:Paragraph|Summary|Answer|Output)\s*:\s*', '', text, flags=re.I)
    if text and text[0] in '"\'\u201c\u2018' and text[-1] in '"\'\u201d\u2019':
        text = text[1:-1].strip()
    return text


def generate_paragraph(slot, params, target, extra_instructions):
    """Generate one paragraph with retries on word-budget and consistency."""
    tol  = TOLERANCE_BY_SLOT.get(slot, TOLERANCE)
    low  = int(round(target * (1 - tol)))
    high = int(round(target * (1 + tol)))

    ev_block  = _evidence_block(params, LABELS['current'], LABELS['forecast'])
    reference = REFERENCE_EXAMPLES[slot]
    base_user = (
        f"EVIDENCE (pre-computed regional patterns for {LABELS['current']} {YEAR} "
        f"and the {LABELS['forecast']} forecast):\n{ev_block}\n\n"
        f"REFERENCE PARAGRAPH (a paragraph from a previous month's outlook — "
        f"match this writing style, but DO NOT copy its specific regional claims; "
        f"the regional patterns may be different this month):\n"
        f"\"{reference}\"\n\n"
        f"WRITE: one paragraph for the section \"{slot}\". {extra_instructions}\n\n"
        f"TARGET LENGTH: {target} words (between {low} and {high} inclusive). "
        f"Mention conditions in {LABELS['current']} (current) AND in "
        f"{LABELS['forecast']} (forecast).\n\n"
        f"Output ONLY the paragraph text. No labels, no quotes, no markdown."
    )

    candidates = []   # list of (n_contradictions, word_count_distance, text)
    for attempt in range(1, MAX_ATTEMPTS + 1):
        if attempt == 1:
            user = base_user
        else:
            # Build a corrective rider based on the best-so-far problems
            best = min(candidates, key=lambda c: (c[0], c[1]))
            feedback_parts = []
            if best[1] > 0:
                wc = word_count(best[2])
                if wc > high: feedback_parts.append(f"shorten (last try was {wc} words, target {target})")
                else:         feedback_parts.append(f"lengthen (last try was {wc} words, target {target})")
            if best[0] > 0:
                feedback_parts.append(f"the last try had {best[0]} statements that contradict the evidence — "
                                       "carefully re-read the EVIDENCE block and only describe directions it supports")
            rider = "; ".join(feedback_parts)
            user = base_user + f"\n\n(Previous attempt feedback: {rider}.)"

        _, content = BACKEND.chat(SYSTEM_PROMPT, user)
        text = _strip_wrapping(content)
        wc = word_count(text)
        wd = 0 if low <= wc <= high else min(abs(wc - low), abs(wc - high))
        ncontra, _ = score_paragraph(text, EVIDENCE, time_key='current')
        candidates.append((ncontra, wd, text))

        # Early exit if perfect
        if ncontra == 0 and wd == 0:
            return text, wc, attempt, 0

    # Pick the best candidate: prioritise zero contradictions, then word-budget fit
    candidates.sort(key=lambda c: (c[0], c[1]))
    best = candidates[0]
    return best[2], word_count(best[2]), MAX_ATTEMPTS, best[0]


# -----------------------------------------------------------------------
# The 12 paragraph specs
# -----------------------------------------------------------------------
PARAGRAPH_SPEC = [
    ('page1_summary',           ['P','T','sm','ro','ET','Q','Station_Q'],
        "This is the top-of-page-1 SUMMARY paragraph. Cover ALL seven parameters in one flowing paragraph: rainfall, temperature, relative wetness, total runoff, evapotranspiration, streamflow in rivers and reaches, and streamflow at gauge stations. Order: current-month observations first, then the forecast."),
    ('page1_rainfall_temp',     ['P','T'],
        "This is the 'Rainfall and Temperature' section on page 1. Two sentences on rainfall (current + forecast) and two on temperature (current + forecast)."),
    ('page1_sm_ro_et',          ['sm','ro','ET'],
        "This is the 'Soil moisture, Total runoff, and Evapotranspiration' section on page 1. Cover each of the three sub-topics; mention both current and forecast for each."),
    ('page1_rivers',            ['Q','Station_Q'],
        "This is the 'River flows' section on page 1. Combine the two streamflow parameters (network reaches + gauge stations). Current first, forecast second."),
    ('page2_rainfall_yellow',   ['P'],
        "This is the yellow Summary banner on the Rainfall page. Brief: one or two short sentences. Cover both current and forecast."),
    ('page3_temperature_yellow',['T'],
        "This is the yellow Summary banner on the Temperature page. Brief; current and forecast."),
    ('page4_wetness_yellow',    ['sm'],
        "This is the yellow Summary banner on the Relative Wetness page. Brief; current and forecast."),
    ('page5_runoff_yellow',     ['ro'],
        "This is the yellow Summary banner on the Total Runoff page. Brief; current and forecast."),
    ('page6_et_yellow',         ['ET'],
        "This is the yellow Summary banner on the Evapotranspiration page. Brief; current and forecast."),
    ('page7_stationq_yellow',   ['Station_Q'],
        "This is the yellow Summary banner on the gauge-station streamflow page. Brief; current and forecast."),
    ('page8_networkq_yellow',   ['Q'],
        "This is the yellow Summary banner on the stream-network streamflow page. Brief; current and forecast."),
]

print(f'Generating {len(PARAGRAPH_SPEC)} paragraphs '
      f'(target ±{int(TOLERANCE*100)}% word budget, contradiction-checked, '
      f'up to {MAX_ATTEMPTS} retries each)…\n')

PARAGRAPHS = {}
for slot, params, extra in PARAGRAPH_SPEC:
    target = TARGETS[slot]
    tol    = TOLERANCE_BY_SLOT.get(slot, TOLERANCE)
    text, wc, attempts, ncontra = generate_paragraph(slot, params, target, extra)
    PARAGRAPHS[slot] = text
    wc_ok = abs(wc - target) <= target * tol
    co_ok = ncontra == 0
    status = '✓' if (wc_ok and co_ok) else '⚠️'
    flags = []
    if not wc_ok: flags.append(f'wc={wc} (target {target}, tol ±{int(tol*100)}%)')
    if not co_ok: flags.append(f'contradictions={ncontra}')
    flag_str = '  '.join(flags) if flags else 'clean'
    print(f'  {status} {slot:30s}  attempts={attempts}  {flag_str}')
    snippet = text[:130] + ('…' if len(text) > 130 else '')
    print(f'      {snippet}\n')


## 10 · Inspect the generated paragraphs (optional)

Print all 12 paragraphs side-by-side for a sanity check before we commit them to the PDF.

In [ ]:
print('=' * 80)
for slot, _, _ in PARAGRAPH_SPEC:
    print(f'\n[{slot}]  ({word_count(PARAGRAPHS[slot])} words, target {TARGETS[slot]})')
    print(PARAGRAPHS[slot])
    print('-' * 80)

## 11 · Build the LaTeX source and compile the PDF

We hold the template in the notebook as a Python string so the pipeline is self-contained. Substitutions are done with `{{slot_name}}` placeholders so there's zero risk of mangling actual LaTeX braces.

We also copy the freshly-generated dashboards and the static PDF_images into a working `images/` directory next to the `.tex` so `\graphicspath{{images/}}` resolves correctly.

In [ ]:
# ----- LaTeX template -----
# Same structure as the manually-written hydrolook.tex. The 12 paragraphs use
# {{slot}} placeholders. The image filenames stay literal — we'll just put the
# right files at those paths.
#
# Side ribbon / page header references {{RIBBON_DATE}} so the ribbon updates each month.
TEX_TEMPLATE = r'''
% ============================================================
% India Hydrological Outlook - AUTO-GENERATED
% Page size: 20x20 inches
% ============================================================
\documentclass[20pt]{extarticle}

\usepackage[paperwidth=20in, paperheight=20in, margin=0in]{geometry}
\usepackage{graphicx}
\usepackage[table]{xcolor}
\usepackage{tikz}
\usepackage{tcolorbox}
\usepackage{ragged2e}
\usepackage{xurl}
\usepackage{hyperref}
\usepackage{array}
\usepackage{fontspec}
\setmainfont{Carlito}
\setsansfont{Carlito}
\renewcommand{\familydefault}{\sfdefault}
\usepackage{microtype}
\tcbuselibrary{skins}
\usetikzlibrary{positioning, calc}

\graphicspath{{images/}}

\definecolor{titleblue}{HTML}{0F2F4A}
\definecolor{textblack}{HTML}{1A1A1A}
\definecolor{summarygold}{HTML}{F5E9A8}
\definecolor{summarygoldborder}{HTML}{4D7A38}
\definecolor{sidebandorange}{HTML}{E89640}
\definecolor{footerblue}{HTML}{B7D3E5}
\definecolor{linkblue}{HTML}{1A5BAA}

\hypersetup{colorlinks=true, urlcolor=linkblue, linkcolor=linkblue}
\setlength{\parindent}{0pt}
\pagestyle{empty}
\hbadness=10000 \vbadness=10000

\newcommand{\pageribbon}{%
\begin{tikzpicture}[remember picture, overlay]
  \fill[sidebandorange] ([xshift=-0.85in]current page.north east) rectangle
                        ([xshift=0in]current page.south east);
  \node[rotate=-90, text=white, font=\Huge\bfseries, anchor=center]
    at ([xshift=-0.42in,yshift=-4.0in]current page.north east) {{{RIBBON_DATE}}};
  \node[rotate=-90, text=white, font=\Huge\bfseries, anchor=center]
    at ([xshift=-0.42in,yshift=-13.5in]current page.north east) {India Hydrological Outlook};
\end{tikzpicture}%
}

\newcommand{\pageheader}[1]{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.4,-0.25) {%
    \includegraphics[height=1.4in]{WCL_Logo_cropped.png}%
  };
  \node[anchor=center, text=titleblue, font=\Large\bfseries, align=center, text width=14in]
    at (9.5,-0.55) {#1};
  \node[anchor=center, text=textblack, font=\normalsize, align=center]
    at (9.5,-1.05) {Based on daily observations till {{OBSERVATION_DATE}}};
  \node[anchor=east, text=textblack, font=\normalsize] at (18.95,-1.05) {Issue date: {{ISSUE_DATE}}};
\end{tikzpicture}%
}

\newcommand{\pagenum}[1]{%
\begin{tikzpicture}[remember picture, overlay]
  \node[text=white, font=\LARGE\bfseries, anchor=center]
    at ([xshift=-0.42in,yshift=0.5in]current page.south east) {#1};
\end{tikzpicture}%
}

\newcommand{\pagefooterbanner}{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.south west)]
  \node[anchor=south west, inner sep=0pt] at (0.5,0.5) {%
    \begin{minipage}{18.4in}
      \begin{tcolorbox}[colback=footerblue, colframe=footerblue, boxrule=0.8pt, arc=8pt,
                        left=18pt, right=18pt, top=14pt, bottom=14pt]
      {\normalsize\textcolor{textblack}{India Hydrological Outlook provides a comprehensive monthly summary of key meteorological and hydrological variables for current conditions alongside a four-month retrospective and one month forecast. For more information, please visit the website:~\href{http://www.indiahydrolook.in}{www.indiahydrolook.in}}}
      \end{tcolorbox}
    \end{minipage}%
  };
\end{tikzpicture}%
}

\newcommand{\pagefooterfull}{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.south west)]
  \node[anchor=south west, inner sep=0pt] at (0.5,2.05) {%
    \begin{minipage}{18.4in}
      \begin{tcolorbox}[colback=footerblue, colframe=footerblue, boxrule=0.8pt, arc=8pt,
                        left=18pt, right=18pt, top=14pt, bottom=14pt]
      {\normalsize\textcolor{textblack}{India Hydrological Outlook provides a comprehensive monthly summary of key meteorological and hydrological variables for current conditions alongside a four-month retrospective and one month forecast. For more information, please visit the website:~\href{http://www.indiahydrolook.in}{www.indiahydrolook.in}}}
      \end{tcolorbox}
    \end{minipage}%
  };
  \node[anchor=west, inner sep=0pt] at (0.5,1.0) {\includegraphics[height=1.55in]{IITGN.png}};
  \node[anchor=west, inner sep=0pt, text=textblack] at (2.3,1.0) {%
    \begin{tabular}{@{}l@{}}
      {\large\textbf{\textcolor{titleblue}{IIT Gandhinagar}}}\\[3pt]
      {\normalsize\textcolor{textblack}{Indian Institute of}}\\[2pt]
      {\normalsize\textcolor{textblack}{Technology Gandhinagar}}
    \end{tabular}};
  \node[anchor=center, inner sep=0pt] at (9.5,1.0) {\includegraphics[height=1.6in]{WCL_Logo_cropped.png}};
  \node[anchor=east, inner sep=0pt] at (18.85,1.0) {\includegraphics[height=1.75in]{IMD_nobg.png}};
\end{tikzpicture}%
}

\newcommand{\summaryat}[2]{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.5,-#1) {%
    \begin{minipage}{18.4in}
      \begin{tcolorbox}[colback=summarygold, colframe=summarygoldborder,
                        boxrule=0pt, leftrule=3pt, arc=0pt, sharp corners,
                        left=14pt, right=14pt, top=10pt, bottom=10pt]
      {\normalsize\textbf{Summary:} #2}
      \end{tcolorbox}
    \end{minipage}%
  };
\end{tikzpicture}%
}

\newcommand{\summaryone}[2]{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.5,-#1) {%
    \begin{minipage}{18.4in}
      \begin{tcolorbox}[colback=footerblue, colframe=footerblue, boxrule=0.8pt, arc=8pt,
                        left=18pt, right=18pt, top=14pt, bottom=14pt]
      {\normalsize\textcolor{textblack}{\textbf{SUMMARY:} #2}}
      \end{tcolorbox}
    \end{minipage}%
  };
\end{tikzpicture}%
}

\newcommand{\introat}[2]{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.5,-#1) {%
    \begin{minipage}{18.4in}\justifying\normalsize\textcolor{textblack}{#2}\end{minipage}};
\end{tikzpicture}%
}

\newcommand{\imageat}[3]{%
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north, inner sep=0pt] at (9.5,-#1) {\includegraphics[width=18.3in,height=#2in,keepaspectratio]{#3}};
\end{tikzpicture}%
}

\newcommand{\circphoto}[2]{\includegraphics[width=#2in]{#1}}

\begin{document}

% ========== PAGE 1 ==========
\pageribbon
\pageheader{India Hydrological Outlook}
\summaryone{1.7}{{{page1_summary}}}
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.5,-5.9) {%
    \begin{minipage}[t]{8.2in}\vspace{0pt}
      {\textbf{\textcolor{titleblue}{\Large Rainfall and Temperature:}}}\\[0.5em]
      {\justifying\normalsize {{page1_rainfall_temp}}\par}
      \vspace{1.4em}
      {\textbf{\textcolor{titleblue}{\Large Soil moisture, Total runoff, and Evapotranspiration:}}}\\[0.5em]
      {\justifying\normalsize {{page1_sm_ro_et}}\par}
      \vspace{1.4em}
      {\textbf{\textcolor{titleblue}{\Large River flows:}}}\\[0.5em]
      {\justifying\normalsize {{page1_rivers}}\par}
    \end{minipage}};
  \node[anchor=north west, inner sep=0pt] at (9.2,-5.9) {%
    \begin{minipage}[t]{9.5in}\vspace{0pt}\centering
      \includegraphics[width=\linewidth]{1-s2_0-S0022169421010271-gr1_lrg.png}\\[0.5em]
      {\justifying\normalsize\textbf{Figure.} Indian sub-continental river basins. Shaded background shows elevation (m).}
    \end{minipage}};
\end{tikzpicture}
\pagefooterfull\pagenum{1}\null\clearpage

% ========== PAGE 2 — Rainfall ==========
\pageribbon\pageheader{Observed and Forecast Rainfall Conditions}
\introat{1.9}{These maps are prepared from daily gridded rainfall observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as total monthly rainfall in percentile for period 1955 to present. These rainfall percentile maps highlight the regions with unusual dry, wet, or near-average rainfall for that time of year. High rainfall percentiles in forecast month, along with high relative wetness (maps available on page 4) in current month, may indicate a potential region for unusual wet conditions. Areas with low precipitation percentiles in forecast month, particularly those with minimal or no rainfall along with high relative dryness, may be at risk of drought conditions.}
\summaryat{4.3}{{{page2_rainfall_yellow}}}
\imageat{5.7}{12.5}{Rainfall_dashboard.png}
\pagefooterbanner\pagenum{2}\null\clearpage

% ========== PAGE 3 — Temperature ==========
\pageribbon\pageheader{Observed and Forecast Surface Air Temperature}
\introat{1.9}{These maps are prepared from daily gridded temperature observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as a monthly-averaged temperature anomaly from the historical mean (1955 to present). These temperature anomaly maps highlight regions with higher and lower temperatures than usual. Low-temperature anomalies may lead to a decrease in crop yield or delayed crop growth. Areas with higher temperature anomalies paired with relatively low rainfall and low soil moisture (maps available on page 4) may be at risk of drought conditions with increased potential evapotranspiration.}
\summaryat{4.3}{{{page3_temperature_yellow}}}
\imageat{5.7}{12.5}{Temperature_dashboard.png}
\pagefooterbanner\pagenum{3}\null\clearpage

% ========== PAGE 4 — Relative Wetness ==========
\pageribbon\pageheader{Observed and Forecast Relative Wetness}
\introat{1.9}{These maps are prepared from daily simulated soil moisture (60 cm depth) using gridded meteorological forcing from observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as a monthly-averaged soil moisture anomaly from the historical mean (1955 to present). The soil moisture anomalies are presented as relative to historical extremes to highlight the regions with unusual wet or dry conditions. High relative wetness (wetter) in current month paired with a high rainfall percentile in forecast month may lead to above-normal flows and unusual wetter conditions in the coming days/weeks. Areas with low relative wetness (drier) in current month and no or minimal predicted rainfall in forecast month may be at risk of lower than normal flow and moisture availability.}
\summaryat{4.3}{{{page4_wetness_yellow}}}
\imageat{5.7}{12.5}{Relative_Wetness_dashboard.png}
\pagefooterbanner\pagenum{4}\null\clearpage

% ========== PAGE 5 — Total Runoff ==========
\pageribbon\pageheader{Observed and Forecast Total Runoff}
\introat{1.9}{These maps are prepared from daily simulated total runoff using gridded meteorological forcing from observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as a total runoff monthly anomaly from the historic mean (1955 to present). These total runoff anomalies highlight the regions with relative surplus or deficit in total runoff than typical for that time of year. A surplus in total runoff in forecast month paired with relatively higher soil moisture in current month may lead to above-normal streamflow in the coming days/weeks. Areas with a high deficit in total runoff in forecast month, particularly those with high dryness in current month, may be at risk of relatively low streamflow in the coming days/weeks.}
\summaryat{4.3}{{{page5_runoff_yellow}}}
\imageat{5.7}{12.5}{Total_Runoff_dashboard.png}
\pagefooterbanner\pagenum{5}\null\clearpage

% ========== PAGE 6 — Evapotranspiration ==========
\pageribbon\pageheader{Observed and Forecast Evapotranspiration}
\introat{1.9}{These maps are prepared from daily simulated evapotranspiration (ET) using gridded meteorological forcing from observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as a total monthly ET anomaly from the historic (1955 to present). These ET anomalies highlight the regions with less or more water lost than usual from land and plants, helping to pinpoint areas at risk of extreme weather impacts, especially crop water stress. Low ET anomalies indicate increased crop water stress and relatively less moisture availability. High ET anomalies may lead to rapid depletion of soil moisture (flash drought) or may worsen the pre-existed droughts.}
\summaryat{4.3}{{{page6_et_yellow}}}
\imageat{5.7}{12.5}{Evapotranspiration_dashboard.png}
\pagefooterbanner\pagenum{6}\null\clearpage

% ========== PAGE 7 — Streamflow at Gauge Stations ==========
\pageribbon\pageheader{Observed and Forecast Streamflow at Gauge Stations}
\introat{1.9}{These maps are prepared from daily simulated streamflow at 225 stations using gridded meteorological forcing from observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as monthly-averaged streamflow in percentile for period 1955 to present. These streamflow percentile maps highlight the locations with unusually high and low flows. A higher streamflow percentile in current month paired with a higher rainfall percentile and higher relative wetness in forecast month may indicate potential regions for extremely high flows in the coming days/weeks. Areas with low streamflow percentiles in current month, along with low rainfall percentile and low relative wetness in forecast month, may be at risk of extreme low flows or hydrological drought conditions.}
\summaryat{4.3}{{{page7_stationq_yellow}}}
\imageat{5.7}{12.5}{Streamflow_Stations_dashboard.png}
\pagefooterbanner\pagenum{7}\null\clearpage

% ========== PAGE 8 — Streamflow at Stream Network ==========
\pageribbon\pageheader{Observed and Forecast Streamflow at Stream Network}
\introat{1.9}{These maps are prepared from daily simulated streamflow in river reaches using gridded meteorological forcing from observations and Extended Range Forecast System (ERFS) from India Meteorological Department (IMD), expressed as monthly-averaged streamflow in percentile for period 1955 to present. These streamflow percentile maps highlight the streams with unusually high and low flows. A higher streamflow percentile in current month paired with a higher rainfall percentile and higher relative wetness in forecast month may indicate potential regions for extremely high flows in the coming days/weeks. Areas with low streamflow percentiles in current month, along with low rainfall percentile and low relative wetness in forecast month, may be at risk of extreme low flows or hydrological drought conditions.}
\summaryat{4.3}{{{page8_networkq_yellow}}}
\imageat{5.7}{12.5}{Streamflow_Network_dashboard.png}
\pagefooterbanner\pagenum{8}\null\clearpage

% ========== PAGE 9 — About / Disclaimer / Contact (STATIC) ==========
\pageribbon\pageheader{India Hydrological Outlook}
\begin{tikzpicture}[remember picture, overlay, x=1in, y=1in, shift=(current page.north west)]
  \node[anchor=north west, inner sep=0pt] at (0.5,-2.2) {%
    \begin{minipage}[t]{9.0in}\vspace{0pt}
      {\textcolor{titleblue}{\LARGE About the India Hydrological Outlook}}\\[0.5em]
      {\justifying\normalsize\textcolor{textblack}{This document provides a comprehensive water outlook for India, covering the current month, the past four months, and a one-month forecast. The outlook is generated using observational data and an extended-range forecasting system for meteorological variables, combined with advanced hydrological modeling tools. These tools analyze relative changes and predict water availability across various regions, offering valuable insights into hydrological situation in India. Developed by the Water and Climate Lab at IIT Gandhinagar, this outlook aims to support water resource management, planning, and decision-making by offering reliable information on rainfall, temperature, soil moisture (60 cm), runoff, and streamflow patterns across different timescales.}\par}
      \vspace{1.3em}
      {\textcolor{titleblue}{\LARGE Datasets}}\\[0.5em]
      {\justifying\normalsize\textcolor{textblack}{The India Hydrological Outlook is made possible through the valuable cooperation of numerous data providers, whose contributions are gratefully acknowledged. Contemporary daily observations and extended range forecast data are supplied by the India Meteorological Department (IMD)~\href{https://www.imdpune.gov.in/cmpg/Griddata/Rainfall_25_Bin.html}{https://www.imdpune.gov.in/cmpg/Griddata/Rainfall\_25\_Bin.html}. These datasets are used for initializing hydrological models and generating reliable outlook information through statistical analysis of historical analogues. Hydrological model is calibrated and validated using daily streamflow for sufficient long duration from India-Water Information System~\href{https://indiawris.gov.in/wris\#RiverMonitoring}{https://indiawris.gov.in/wris\#RiverMonitoring}.}\par}
      \vspace{1.3em}
      {\textcolor{titleblue}{\LARGE Hydrological Model}}\\[0.5em]
      {\justifying\normalsize\textcolor{textblack}{The India Hydrological Outlook utilizes the Variable Infiltration Capacity (VIC) model, a large-scale, semi-distributed hydrological model that computes water and energy budgets for each grid. The VIC model captures sub-grid variability in vegetation and elevation, providing a more accurate representation of hydrological processes at a regional scale. VIC model simulations are performed at a daily temporal resolution to provide accurate hydrological forecasts and enhance water resource management across India.}\par}
    \end{minipage}};
  \node[anchor=north west, inner sep=0pt] at (9.9,-2.2) {%
    \begin{minipage}[t]{8.9in}\vspace{0pt}
      {\textcolor{titleblue}{\LARGE Disclaimer and Liability}}\\[0.5em]
      {\justifying\normalsize\textcolor{textblack}{The India Hydrological Outlook aims to ensure that all content provided is accurate and aligns with the current scientific understanding. However, the science underlying the meteorological and hydrological forecasts, as well as climate projections, is continuously evolving. As such, any forecast or prediction included in the content should not be regarded as a definitive statement of fact. To the fullest extent permitted by applicable law, the India Hydrological Outlook disclaims all warranties or representations, whether express or implied, regarding the content. Your use of the content is entirely at your own risk, and we make no guarantees that the content is error-free or suitable for your specific needs.}\par}
      \vspace{0.7em}
      {\justifying\normalsize\textcolor{textblack}{The India Hydrological Outlook is supported by the Major Research and Development Programme (MRDP) in Hydro Climate Extremes, funded by the Department of Science and Technology (DST).}\par}
      \vspace{1.3em}
      {\textcolor{titleblue}{\LARGE Contact Information}}\\[0.5em]
      {\normalsize\textcolor{textblack}{India Hydrological Outlook, Water and Climate Lab, Indian Institute of Technology Gandhinagar, Gujarat, India.~\href{http://www.indiahydrolook.in}{Water \& Climate Lab}}\par}
      \vspace{0.4em}
      {\normalsize\textcolor{textblack}{India Hydrological Outlook Portal:~\href{http://www.indiahydrolook.in}{www.indiahydrolook.in}}\par}
      \vspace{1.0em}
      \begin{tabular}{@{}m{2.0in}@{\hspace{0.3in}}m{6.5in}@{}}
      \circphoto{prof_vimal_circle.png}{1.9} &
      \normalsize\textcolor{textblack}{Prof. Vimal Mishra (Professor)\newline
      Department of Civil Engineering, IIT Gandhinagar\newline
      \textbf{Email}: vmishra@iitgn.ac.in\newline
      \textbf{Office}: AB6/330, IIT Gandhinagar}
      \end{tabular}
      \vspace{1.2em}
      \begin{tabular}{@{}>{\centering\arraybackslash}m{4.2in}@{\hspace{0.3in}}>{\centering\arraybackslash}m{4.2in}@{}}
      \circphoto{devesh_circle.png}{1.9} &
      \circphoto{paras_circle.png}{1.9} \\[0.4em]
      \normalsize\textcolor{textblack}{Devesh Mani\newline PhD Research Scholar\newline IIT Gandhinagar\newline 24350007@iitgn.ac.in} &
      \normalsize\textcolor{textblack}{Paras Sharma\newline PhD Research Scholar\newline IIT Gandhinagar\newline paras.sharma@iitgn.ac.in}
      \end{tabular}
    \end{minipage}};
\end{tikzpicture}
\pagefooterfull\pagenum{9}\null

\end{document}
'''

print(f'✓ template loaded ({len(TEX_TEMPLATE)} chars)')

In [ ]:
# ----- Build the working directory and substitute placeholders -----
BUILD_DIR  = Path('/content/pdf_build')
IMAGES_DIR = BUILD_DIR / 'images'
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
IMAGES_DIR.mkdir(parents=True)

# Static images shipped from Drive
STATIC_FILES = ['WCL_Logo_cropped.png', 'IITGN.png', 'IMD_nobg.png',
                '1-s2_0-S0022169421010271-gr1_lrg.png',
                'prof_vimal_circle.png', 'devesh_circle.png', 'paras_circle.png']
for fn in STATIC_FILES:
    src = STATIC_IMG_DIR / fn
    if not src.exists():
        raise FileNotFoundError(f'Static image missing on Drive: {src}')
    shutil.copy(src, IMAGES_DIR / fn)

# Freshly-generated dashboards from this run
DASH_FILES = ['Rainfall_dashboard.png','Temperature_dashboard.png','Relative_Wetness_dashboard.png',
              'Total_Runoff_dashboard.png','Evapotranspiration_dashboard.png',
              'Streamflow_Stations_dashboard.png','Streamflow_Network_dashboard.png']
for fn in DASH_FILES:
    shutil.copy(DASHBOARDS_DIR / fn, IMAGES_DIR / fn)

# LaTeX special-character escaping for prose (paragraph contents)
def latex_escape(s):
    if s is None: return ''
    repl = [('\\', r'\textbackslash{}'), ('&', r'\&'), ('%', r'\%'),
            ('$', r'\$'), ('#', r'\#'), ('_', r'\_'),
            ('{', r'\{'), ('}', r'\}'), ('~', r'\textasciitilde{}'),
            ('^', r'\textasciicircum{}')]
    for a, b in repl: s = s.replace(a, b)
    return s

# Issue date = day after the snapshot date — published the next morning
from datetime import timedelta
snap = datetime(YEAR, MONTH, DAY)
issue = snap + timedelta(days=1)
OBSERVATION_DATE_FMT = snap.strftime('%-dth %B %Y') if hasattr(snap, 'strftime') else f'{DAY}th {LABELS["current"]} {YEAR}'
# Make the suffix correct (1st/2nd/3rd/etc.)
def _ordinal(d):
    if 10 <= d % 100 <= 20: suf = 'th'
    else: suf = {1:'st',2:'nd',3:'rd'}.get(d % 10, 'th')
    return f'{d}{suf}'
OBSERVATION_DATE = f'{_ordinal(DAY)} {LABELS["current"]} {YEAR}'
ISSUE_DATE       = f'{issue.day:02d}.{issue.month:02d}.{issue.year}'
RIBBON_DATE      = f'{LABELS["current"]} {YEAR}'

# Substitute placeholders. Order matters: do header tokens first, then
# paragraph slots (some headers contain the literal text 'page' which is harmless).
tex = TEX_TEMPLATE
tex = tex.replace('{{RIBBON_DATE}}',      RIBBON_DATE)
tex = tex.replace('{{OBSERVATION_DATE}}', OBSERVATION_DATE)
tex = tex.replace('{{ISSUE_DATE}}',       ISSUE_DATE)
for slot, text in PARAGRAPHS.items():
    tex = tex.replace('{{' + slot + '}}', latex_escape(text))

tex_path = BUILD_DIR / 'hydrolook.tex'
tex_path.write_text(tex, encoding='utf-8')
print(f'✓ wrote {tex_path}  ({len(tex)} chars)')
print(f'  images dir : {IMAGES_DIR}  ({len(list(IMAGES_DIR.iterdir()))} files)')

In [ ]:
# Compile with XeLaTeX. Run twice so the TikZ 'remember picture / overlay'
# references settle (the second pass uses the .aux from the first).
import subprocess

def run_xelatex():
    cmd = ['xelatex', '-interaction=nonstopmode', '-halt-on-error',
           '-output-directory', str(BUILD_DIR), str(tex_path)]
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=str(BUILD_DIR))
    return res

print('Pass 1 …')
r1 = run_xelatex()
print('  exit:', r1.returncode)
if r1.returncode != 0:
    print(r1.stdout[-3000:])
    print(r1.stderr[-1500:])
    raise RuntimeError('xelatex pass 1 failed — see log above')

print('Pass 2 (settle overlays) …')
r2 = run_xelatex()
print('  exit:', r2.returncode)
if r2.returncode != 0:
    print(r2.stdout[-3000:])
    raise RuntimeError('xelatex pass 2 failed')

out_pdf_local = BUILD_DIR / 'hydrolook.pdf'
if not out_pdf_local.exists():
    raise RuntimeError('xelatex returned 0 but no PDF found')
print(f'\n✓ PDF compiled: {out_pdf_local}  ({out_pdf_local.stat().st_size/1e6:.1f} MB)')

In [ ]:
# Place the final PDF in two locations:
#   1. Output/                            — holds ONLY the latest PDF (older PDFs deleted)
#   2. Output/PDF_Archive/                — accumulates every month's PDF (never deleted)
final_name    = f'Hydrolook_{FORECAST_DATE_STR}.pdf'
ARCHIVE_DIR   = OUTPUT_BASE / 'PDF_Archive'
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Latest-only directory ---
# Delete any existing Hydrolook_*.pdf in Output/ so only this month's remains.
for old_pdf in OUTPUT_BASE.glob('Hydrolook_*.pdf'):
    try:
        old_pdf.unlink()
        print(f'  - removed previous latest:  {old_pdf.name}')
    except OSError as e:
        print(f'  ! could not remove {old_pdf}: {e}')

latest_path = OUTPUT_BASE / final_name
shutil.copy(out_pdf_local, latest_path)
print(f'  ✓ wrote latest PDF:           {latest_path}')

# --- 2. Permanent archive ---
archive_path = ARCHIVE_DIR / final_name
shutil.copy(out_pdf_local, archive_path)
print(f'  ✓ wrote archive copy:         {archive_path}')

# Make `final_path` available to subsequent cells (preview, etc.)
final_path = latest_path

print()
print(f'All outputs for {FORECAST_DATE_STR}:')
print(f'  Individual maps : {ALL_MAPS_DIR}')
print(f'  Dashboards      : {DASHBOARDS_DIR}')
print(f'  Latest PDF      : {latest_path}')
print(f'  Archive entry   : {archive_path}')
print(f'  Archive total   : {len(list(ARCHIVE_DIR.glob("Hydrolook_*.pdf")))} PDF(s)')


## 12 · Preview the PDF in the notebook (optional)

In [ ]:
# Show the first page inline
from IPython.display import IFrame, display
try:
    display(IFrame(str(out_pdf_local), width=900, height=900))
except Exception:
    print(f'Open {final_path} in Drive to view.')

## 13 · Monthly re-run

Next month, drop the 7 new files into `Hydrologic_Outlook/Input/` (with the new `YYYY_MM_DD` suffix), then *Runtime → Run all*. The pipeline:

- auto-detects the latest date in the Input folder (Section 1),
- regenerates every map and dashboard from those data files (Sections 5–6),
- recomputes regional stats and asks the LLM for fresh paragraphs (Sections 7–9),
- recompiles the PDF and writes `Hydrolook_<new_date>.pdf` to Drive (Sections 11–12).

Nothing else needs editing.

**Troubleshooting:**
- *Paragraph word-count warnings (⚠️)* — the LLM landed outside the ±15% target. The PDF will still compile; the paragraph may run slightly long/short in its box. Re-run the affected cell to retry (the LLM is sampled, so output varies).
- *`xelatex` fails on missing image* — check that all 7 dashboard PNGs and all 7 static images exist (Section 11 lists them).
- *Out-of-memory on Qwen load* — should be impossible on T4 with the 2B model (only ~4 GB needed), but if it happens, fall back to `Qwen/Qwen3-1.7B` or switch to `dtype='bfloat16'` if your runtime supports it.

- *Slow compilation* — the first run installs TeX Live and downloads the model (~8 GB). Subsequent runs in the same Colab session are much faster.